In [7]:
!pip install transformer_lens

In [8]:
"""
milestone3_predictive_benchmark_full.py
======================================
Milestone 3 - Predictive Self-Model Benchmark Engine.
1. Centralized physical and statistical thresholds.
2. Builds Epistemic Evidence Graph (Milestone 2.2 pipeline).
3. Uses SelfModelPredictor to predict multi-node joint interventions (L9H9 + L10H7 -> L11H10).
4. Evaluates Prediction Error (MAE) against ground-truth model outputs.
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# =============================================================================
# Global Configuration & Physical Thresholds
# =============================================================================
RAW_INFLUENCE_FLOOR: float = 0.0100      # حداقل اثر فیزیکی خام
MEDIATION_EFFECT_FLOOR: float = 0.0100   # حداقل اثر واسطه‌ای
STATISTICAL_P_THRESHOLD: float = 0.05    # آستانه معناداری آماری (Empirical P-value)
DEFAULT_K_NULL_SAMPLES: int = 64         # تعداد نمونه‌های توزیع نویز


# =============================================================================
# 1. Epistemic Evidence Schema
# =============================================================================

@dataclass
class InterventionEvidence:
    intervention_type: str
    source: Tuple[int, int]
    target: Tuple[int, int]
    intervention_site: str

    raw_effect: Optional[float] = None
    null_mean: Optional[float] = None
    null_std: Optional[float] = None
    null_z: Optional[float] = None
    empirical_p: Optional[float] = None
    null_variance_status: str = "VALID"

    target_delta: Optional[float] = None
    output_delta: Optional[float] = None

    natural_effect: Optional[float] = None
    clamped_effect: Optional[float] = None
    mediated_effect: Optional[float] = None
    mediation_ratio: Optional[float] = None
    direction_label: str = "NEUTRAL"

    n_null_samples: int = 0
    metadata: Dict = field(default_factory=dict)


@dataclass
class EvidenceLadder:
    level1_target_value: Optional[InterventionEvidence] = None
    level2_residual_path: Optional[InterventionEvidence] = None
    level3_attention_effect: Optional[InterventionEvidence] = None

    level4_qkv_attribution: Optional[Dict] = None
    level5_mechanistic_edge: Optional[Dict] = None


@dataclass
class EvidenceEdge:
    source: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str
    evidence: EvidenceLadder = field(default_factory=EvidenceLadder)
    metadata: Dict = field(default_factory=dict)

    @property
    def status(self) -> str:
        ev = self.evidence

        l1 = ev.level1_target_value
        if l1 is None or abs(l1.raw_effect or 0) < RAW_INFLUENCE_FLOOR:
            return "CANDIDATE"

        l1_valid = (l1.empirical_p is not None and l1.empirical_p <= STATISTICAL_P_THRESHOLD) or \
                   (l1.null_z is not None and abs(l1.null_z) >= 3.0)

        if not l1_valid:
            return "CANDIDATE"

        if ev.level2_residual_path is None:
            return "TARGET_VALUE_INFLUENCE"

        l3 = ev.level3_attention_effect
        if l3 is not None and abs(l3.mediated_effect or 0) >= MEDIATION_EFFECT_FLOOR:
            return "INTERVENTIONAL_ATTENTION_EFFECT"

        return "RESIDUAL_PATH_INFLUENCE"


@dataclass
class SelfCircuitGraph:
    regime_id: str
    nodes: Set[Tuple[int, int]] = field(default_factory=set)
    edges: List[EvidenceEdge] = field(default_factory=list)

    def add_edge(self, edge: EvidenceEdge):
        self.nodes.add(edge.source)
        self.nodes.add(edge.target)
        self.edges.append(edge)

    def print_circuit_summary(self, n_tasks: int):
        print("\n==================================================")
        print(f"🕸️ SELFCIRCUITGRAPH (EPISTEMIC EVIDENCE) - REGIME '{self.regime_id}'")
        print(f"   Note: Evidence evaluated on {n_tasks} task(s). No broad generalization claimed.")
        print("==================================================")
        print(f"Total Candidate Nodes Evaluated: {len(self.nodes)}")
        print(f"Total Tracked Edges: {len(self.edges)}\n")

        for edge in self.edges:
            ev = edge.evidence

            l1 = ev.level1_target_value
            if l1:
                z_str = f"{l1.null_z:+5.2f}" if l1.null_z is not None else f"[{l1.null_variance_status}]"
                p_str = f"{l1.empirical_p:.3f}" if l1.empirical_p is not None else "N/A"
                l1_str = f"Raw: {l1.raw_effect:+6.4f} | Null Mean: {l1.null_mean:+6.4f} (Std: {l1.null_std:.6f}) | Z: {z_str} | Emp p: {p_str}"
            else:
                l1_str = "N/A"

            l2 = ev.level2_residual_path
            l2_str = f"Target Δz: {l2.target_delta:+6.4f} | Output Δ: {l2.output_delta:+6.4f}" if l2 else "N/A"

            l3 = ev.level3_attention_effect
            if l3:
                ratio_str = f"{l3.mediation_ratio:+6.2f}" if l3.mediation_ratio is not None else "N/A"
                l3_str = (f"Natural: {l3.natural_effect:+6.4f} | Clamped: {l3.clamped_effect:+6.4f} | "
                          f"Mediated Δ: {l3.mediated_effect:+6.4f} | Ratio: {ratio_str} | Direction: [{l3.direction_label}]")
            else:
                l3_str = "N/A"

            print(f"  ★ Pair L{edge.source[0]}H{edge.source[1]:<2} ──► L{edge.target[0]}H{edge.target[1]:<2}")
            print(f"     Status: [{edge.status}]")
            print(f"     ├── Level 1 (Target Value Influence):     {l1_str}")
            print(f"     ├── Level 2 (Residual Path Influence):     {l2_str}")
            print(f"     ├── Level 3 (Interventional Attn Effect):  {l3_str}")
            print(f"     ├── Level 4 (Q/K/V Attribution):          [NOT TESTED]")
            print(f"     └── Level 5 (Mechanistic Edge):            [NOT ESTABLISHED]\n")
        print("==================================================\n")


# =============================================================================
# 2. Primitive Operations & Pipeline Engine
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_attention_clamping_intervention(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    base_logits: torch.Tensor,
    corrupted_toks: torch.Tensor,
    pos: int = -1
) -> Tuple[float, float, float, float, str]:
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())

    # 1. Natural
    def patch_res_hook(act, hook):
        act[0, pos, :] += delta_contrib
        return act

    with torch.no_grad():
        natural_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", patch_res_hook)])

    natural_diff = float((natural_logits[0, pos, correct_id] - natural_logits[0, pos, incorrect_id]).item())
    natural_effect = natural_diff - base_diff

    # 2. Clamped
    corrupted_pattern = corrupted_cache[f"blocks.{l_tgt}.attn.hook_pattern"][0, h_tgt, :, :].clone()

    def clamp_attention_pattern_hook(act, hook):
        act[0, h_tgt, :, :] = corrupted_pattern
        return act

    clamped_hooks = [
        (f"blocks.{l_tgt}.hook_resid_pre", patch_res_hook),
        (f"blocks.{l_tgt}.attn.hook_pattern", clamp_attention_pattern_hook)
    ]

    with torch.no_grad():
        clamped_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=clamped_hooks)

    clamped_diff = float((clamped_logits[0, pos, correct_id] - clamped_logits[0, pos, incorrect_id]).item())
    clamped_effect = clamped_diff - base_diff

    mediated_effect = natural_effect - clamped_effect
    abs_natural = abs(natural_effect)
    mediation_ratio = (mediated_effect / abs_natural) if abs_natural > 1e-6 else 0.0

    if abs(mediated_effect) < MEDIATION_EFFECT_FLOOR:
        direction_label = "NEUTRAL"
    elif np.sign(natural_effect) == np.sign(mediated_effect):
        direction_label = "AMPLIFYING" if abs(natural_effect) > abs(clamped_effect) else "SUPPRESSIVE"
    else:
        direction_label = "OPPOSING"

    return natural_effect, clamped_effect, mediated_effect, mediation_ratio, direction_label


class EpistemicEvidenceEngine:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()
        self.rng = np.random.default_rng(42)

    def generate_clean_null_distribution(
        self,
        target: Tuple[int, int],
        task_data: List,
        exclude_sources: Set[Tuple[int, int]],
        k_samples: int = DEFAULT_K_NULL_SAMPLES
    ) -> Tuple[float, float, Optional[float], str, List[float]]:
        l_tgt, _ = target
        possible_null_sources = [
            (l, h) for l in range(l_tgt) for h in range(12)
            if (l, h) not in exclude_sources
        ]

        if len(possible_null_sources) < k_samples:
            null_sources = possible_null_sources
        else:
            indices = self.rng.choice(len(possible_null_sources), k_samples, replace=False)
            null_sources = [possible_null_sources[idx] for idx in indices]

        null_deltas = []
        for src in null_sources:
            task_effects = []
            for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                correct_id = int(self.model.to_single_token(task["correct"]))
                incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[target[0], target[1]]

                def patch_v(act, hook):
                    act[0, -1, target[1], :] += delta_v
                    return act

                with torch.no_grad():
                    patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.attn.hook_v", patch_v)])

                p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                task_effects.append(p_diff - base_diff)
            null_deltas.append(float(np.mean(task_effects)))

        null_mean = float(np.mean(null_deltas))
        null_std = float(np.std(null_deltas))

        if null_std < 1e-6:
            variance_status = "NULL_VARIANCE_TOO_SMALL"
            null_z = None
        else:
            variance_status = "VALID"
            null_z = None

        return null_mean, null_std, null_z, variance_status, null_deltas

    def execute_pipeline(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        k_null_samples: int = DEFAULT_K_NULL_SAMPLES
    ) -> SelfCircuitGraph:

        graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])
        n_tasks = len(tasks)

        print(f"\n⚡️ [EPISTEMIC ENGINE] Running Pipeline for Regime '{regime_id}' across {n_tasks} task(s)")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, base_corrupted_logits, clean_logits))

        for j in range(len(sorted_nodes)):
            tgt = sorted_nodes[j]
            target_candidates = {sorted_nodes[i] for i in range(j) if sorted_nodes[i][0] < tgt[0]}

            for i in range(j):
                src = sorted_nodes[i]
                if src[0] >= tgt[0]:
                    continue

                evidence_ladder = EvidenceLadder()

                # --- Level 1 ---
                raw_effects = []
                for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                    contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                    delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[tgt[0], tgt[1]]

                    def patch_v(act, hook):
                        act[0, -1, tgt[1], :] += delta_v
                        return act

                    with torch.no_grad():
                        patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{tgt[0]}.attn.hook_v", patch_v)])
                    p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                    raw_effects.append(p_diff - base_diff)

                avg_raw = float(np.mean(raw_effects))
                null_mean, null_std, _, var_status, null_deltas = self.generate_clean_null_distribution(
                    target=tgt, task_data=task_data, exclude_sources=target_candidates, k_samples=k_null_samples
                )

                if len(null_deltas) > 0:
                    empirical_p = float((np.sum(np.abs(null_deltas) >= abs(avg_raw)) + 1) / (len(null_deltas) + 1))
                else:
                    empirical_p = None

                computed_z = (avg_raw - null_mean) / null_std if var_status == "VALID" else None

                evidence_ladder.level1_target_value = InterventionEvidence(
                    intervention_type="target_value_intervention", source=src, target=tgt,
                    intervention_site=f"blocks.{tgt[0]}.attn.hook_v", raw_effect=avg_raw,
                    null_mean=null_mean, null_std=null_std, null_z=computed_z,
                    empirical_p=empirical_p, null_variance_status=var_status,
                    n_null_samples=k_null_samples, metadata={"n_tasks": n_tasks}
                )

                # --- Level 2 ---
                l1_valid = (empirical_p is not None and empirical_p <= STATISTICAL_P_THRESHOLD) or (computed_z is not None and abs(computed_z) >= 3.0)

                if l1_valid and abs(avg_raw) >= RAW_INFLUENCE_FLOOR:
                    z_deltas, out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        contrib_corrupt = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                        delta_contrib = contrib_clean - contrib_corrupt

                        def patch_res(act, hook):
                            act[0, -1, :] += delta_contrib
                            return act

                        target_z_patched = [None]
                        def cap_z(act, hook):
                            target_z_patched[0] = act[0, -1, tgt[1], :].clone()
                            return act

                        hooks = [(f"blocks.{tgt[0]}.hook_resid_pre", patch_res), (f"blocks.{tgt[0]}.attn.hook_z", cap_z)]
                        with torch.no_grad():
                            patched_logits = self.model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        b_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
                        p_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

                        z_corrupt = corrupted_cache[f"blocks.{tgt[0]}.attn.hook_z"][0, -1, tgt[1], :]
                        z_deltas.append(float((target_z_patched[0] - z_corrupt).norm().item()))
                        out_deltas.append(p_diff - b_diff)

                    evidence_ladder.level2_residual_path = InterventionEvidence(
                        intervention_type="residual_pre_patch", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre",
                        target_delta=float(np.mean(z_deltas)), output_delta=float(np.mean(out_deltas)),
                        metadata={"n_tasks": n_tasks}
                    )

                    # --- Level 3 ---
                    nat_effs, clamp_effs, med_effs, ratios, directions = [], [], [], [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        nat_e, clamp_e, med_e, ratio, dir_label = evaluate_attention_clamping_intervention(
                            self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks
                        )
                        nat_effs.append(nat_e)
                        clamp_effs.append(clamp_e)
                        med_effs.append(med_e)
                        ratios.append(ratio)
                        directions.append(dir_label)

                    avg_med = float(np.mean(med_effs))
                    avg_nat = float(np.mean(nat_effs))
                    avg_ratio = (avg_med / abs(avg_nat)) if abs(avg_nat) > 1e-6 else 0.0

                    if abs(avg_med) < MEDIATION_EFFECT_FLOOR:
                        overall_dir = "NEUTRAL"
                    elif np.sign(avg_nat) == np.sign(avg_med):
                        overall_dir = "AMPLIFYING" if abs(avg_nat) > abs(float(np.mean(clamp_effs))) else "SUPPRESSIVE"
                    else:
                        overall_dir = "OPPOSING"

                    evidence_ladder.level3_attention_effect = InterventionEvidence(
                        intervention_type="interventional_attention_clamping", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.attn.hook_pattern",
                        natural_effect=avg_nat,
                        clamped_effect=float(np.mean(clamp_effs)),
                        mediated_effect=avg_med,
                        mediation_ratio=avg_ratio,
                        direction_label=overall_dir,
                        metadata={"n_tasks": n_tasks}
                    )

                edge = EvidenceEdge(source=src, target=tgt, regime_id=regime_id, evidence=evidence_ladder)
                graph.add_edge(edge)

        return graph


# =============================================================================
# 3. Milestone 3: Predictive Self-Model Benchmark Engine
# =============================================================================

class SelfModelPredictor:
    """
    موتور پیش‌بینی خودشناس:
    پیش‌بینی اثر مداخلات ترکیبی ندیده صرفاً بر اساس گراف علّی ثبت‌شده (بدون اجرای مدل اصلی).
    """
    def __init__(self, circuit_graph: SelfCircuitGraph):
        self.graph = circuit_graph

    def predict_joint_intervention(self, source_nodes: List[Tuple[int, int]], target_node: Tuple[int, int]) -> float:
        predicted_effect = 0.0
        for src in source_nodes:
            edge = next((e for e in self.graph.edges if e.source == src and e.target == target_node), None)
            if edge and edge.evidence.level3_attention_effect:
                # برآورد اولیه: مجموع برداری اثرات طبیعی (Natural Effect) ثبت‌شده در گراف
                predicted_effect += edge.evidence.level3_attention_effect.natural_effect
        return predicted_effect


def run_predictive_benchmark(
    engine: EpistemicEvidenceEngine,
    predictor: SelfModelPredictor,
    joint_sources: List[Tuple[int, int]],
    target: Tuple[int, int],
    tasks: List[Dict]
) -> Dict[str, float]:
    """
    اجرای بنچمارک پیش‌بینی: مقایسه پیش‌بینی گراف علّی با مداخله هم‌زمان واقعی روی مدل.
    """
    l_tgt, h_tgt = target
    actual_effects = []

    # ۱. پیش‌بینی حاصل از Self-Model
    predicted_effect = predictor.predict_joint_intervention(joint_sources, target)

    # ۲. اجرای مداخله ترکیبی واقعی روی مدل اصلی (Ground Truth)
    for task in tasks:
        clean_toks = engine.model.to_tokens(task["clean"])
        corrupted_toks = engine.model.to_tokens(task["corrupted"])

        with torch.no_grad():
            _, clean_cache = engine.model.run_with_cache(clean_toks)
            base_logits, corrupted_cache = engine.model.run_with_cache(corrupted_toks)

        total_delta_contrib = 0.0
        for src in joint_sources:
            l_src, h_src = src
            contrib_clean = compute_head_contribution(engine.model, clean_cache, l_src, h_src)
            contrib_corrupt = compute_head_contribution(engine.model, corrupted_cache, l_src, h_src)
            total_delta_contrib += (contrib_clean - contrib_corrupt)

        def patch_joint_res_hook(act, hook):
            act[0, -1, :] += total_delta_contrib
            return act

        with torch.no_grad():
            patched_logits = engine.model.run_with_hooks(
                corrupted_toks,
                fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", patch_joint_res_hook)]
            )

        correct_id = int(engine.model.to_single_token(task["correct"]))
        incorrect_id = int(engine.model.to_single_token(task["incorrect"]))
        base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
        patched_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

        actual_effects.append(patched_diff - base_diff)

    actual_mean = float(np.mean(actual_effects))
    mae = float(abs(actual_mean - predicted_effect))

    print("\n==================================================")
    print("🎯 MILESTONE 3: PREDICTIVE SELF-MODEL BENCHMARK RESULTS")
    print("==================================================")
    print(f"Joint Intervention Sources: {joint_sources} ──► Target: {target}")
    print(f"Actual Joint Effect (Ground Truth):   {actual_mean:+6.4f}")
    print(f"Self-Model Predicted Joint Effect:    {predicted_effect:+6.4f}")
    print(f"Mean Absolute Error (MAE):            {mae:6.4f}")
    print("==================================================\n")

    return {
        "actual_joint_effect": actual_mean,
        "predicted_joint_effect": predicted_effect,
        "mae": mae
    }


# =============================================================================
# 4. Main Execution
# =============================================================================

if __name__ == "__main__":
    engine = EpistemicEvidenceEngine("gpt2")

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    # ۱. ساخت Evidence Graph
    evidence_graph = engine.execute_pipeline(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        k_null_samples=DEFAULT_K_NULL_SAMPLES
    )

    evidence_graph.print_circuit_summary(n_tasks=len(IOI_TASKS))

    # ۲. اجرای بنچمارک پیش‌بینی Milestone 3
    predictor = SelfModelPredictor(evidence_graph)

    # سنجش پیش‌بینی برای مداخله هم‌زمان L9H9 و L10H7 روی L11H10
    joint_test_sources = [(9, 9), (10, 7)]
    target_node = (11, 10)

    run_predictive_benchmark(
        engine=engine,
        predictor=predictor,
        joint_sources=joint_test_sources,
        target=target_node,
        tasks=IOI_TASKS
    )

/tmp/ipykernel_974/2966499693.py:228: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [EPISTEMIC ENGINE] Running Pipeline for Regime 'R1_IOI' across 2 task(s)

🕸️ SELFCIRCUITGRAPH (EPISTEMIC EVIDENCE) - REGIME 'R1_IOI'
   Note: Evidence evaluated on 2 task(s). No broad generalization claimed.
Total Candidate Nodes Evaluated: 6
Total Tracked Edges: 14

  ★ Pair L7H9  ──► L8H6 
     Status: [CANDIDATE]
     ├── Level 1 (Target Value Influence):     Raw: +0.0000 | Null Mean: -0.0000 (Std: 0.000016) | Z: +1.04 | Emp p: 0.077
     ├── Level 2 (Residual Path Influence):     N/A
     ├── Level 3 (Interventional Attn Effect):  N/A
     ├── Level 4 (Q/K/V Attribution):          [NOT TESTED]
     └── Level 5 (Mechanistic Edge):            [NOT ESTABLISHED]

  ★ Pair L7H9  ──► L8H10
     Status: [CANDIDATE]
     ├── Level 1 (Target Value Influence):     Raw: -0.0044 | Null Mean: +0.0003 (Std: 0.001922) | Z: -2.44 | Emp p: 0.031
     ├── Level 2 (Residual Path Influence):     N/A
     ├── Level 3 (Interventional Attn Effect): 

In [9]:
"""
milestone3_1_standalone_interaction_engine.py
==============================================
Standalone Implementation of Milestone 3.1 (Order-2 Interaction Engine).
Includes all prerequisites from Milestone 2.2 / 3.0 in a single runnable script.
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# =============================================================================
# Global Configuration & Physical Thresholds
# =============================================================================
RAW_INFLUENCE_FLOOR: float = 0.0100      # حداقل اثر فیزیکی خام
MEDIATION_EFFECT_FLOOR: float = 0.0100   # حداقل اثر واسطه‌ای
STATISTICAL_P_THRESHOLD: float = 0.05    # آستانه معناداری آماری
DEFAULT_K_NULL_SAMPLES: int = 64         # تعداد نمونه‌های توزیع نویز


# =============================================================================
# 1. Epistemic Evidence Schema
# =============================================================================

@dataclass
class InterventionEvidence:
    intervention_type: str
    source: Tuple[int, int]
    target: Tuple[int, int]
    intervention_site: str

    raw_effect: Optional[float] = None
    null_mean: Optional[float] = None
    null_std: Optional[float] = None
    null_z: Optional[float] = None
    empirical_p: Optional[float] = None
    null_variance_status: str = "VALID"

    target_delta: Optional[float] = None
    output_delta: Optional[float] = None

    natural_effect: Optional[float] = None
    clamped_effect: Optional[float] = None
    mediated_effect: Optional[float] = None
    mediation_ratio: Optional[float] = None
    direction_label: str = "NEUTRAL"

    n_null_samples: int = 0
    metadata: Dict = field(default_factory=dict)


@dataclass
class EvidenceLadder:
    level1_target_value: Optional[InterventionEvidence] = None
    level2_residual_path: Optional[InterventionEvidence] = None
    level3_attention_effect: Optional[InterventionEvidence] = None

    level4_qkv_attribution: Optional[Dict] = None
    level5_mechanistic_edge: Optional[Dict] = None


@dataclass
class EvidenceEdge:
    source: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str
    evidence: EvidenceLadder = field(default_factory=EvidenceLadder)
    metadata: Dict = field(default_factory=dict)

    @property
    def status(self) -> str:
        ev = self.evidence

        l1 = ev.level1_target_value
        if l1 is None or abs(l1.raw_effect or 0) < RAW_INFLUENCE_FLOOR:
            return "CANDIDATE"

        l1_valid = (l1.empirical_p is not None and l1.empirical_p <= STATISTICAL_P_THRESHOLD) or \
                   (l1.null_z is not None and abs(l1.null_z) >= 3.0)

        if not l1_valid:
            return "CANDIDATE"

        if ev.level2_residual_path is None:
            return "TARGET_VALUE_INFLUENCE"

        l3 = ev.level3_attention_effect
        if l3 is not None and abs(l3.mediated_effect or 0) >= MEDIATION_EFFECT_FLOOR:
            return "INTERVENTIONAL_ATTENTION_EFFECT"

        return "RESIDUAL_PATH_INFLUENCE"


@dataclass
class SelfCircuitGraph:
    regime_id: str
    nodes: Set[Tuple[int, int]] = field(default_factory=set)
    edges: List[EvidenceEdge] = field(default_factory=list)

    def add_edge(self, edge: EvidenceEdge):
        self.nodes.add(edge.source)
        self.nodes.add(edge.target)
        self.edges.append(edge)


@dataclass
class Order2InteractionEvidence:
    """ثبت شواهد تجربی برهم‌کنش غیرجمع‌پذیر دو منبع بر روی یک هدف."""
    source_a: Tuple[int, int]
    source_b: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str

    delta_a: float
    delta_b: float
    delta_ab: float

    additive_prediction: float
    interaction_term: float      # I_AB = Delta_AB - Delta_A - Delta_B
    relative_interaction: float  # I_rel = |I_AB| / (|Delta_AB| + eps)

    interaction_type: str        # SYNERGISTIC, CANCELLING, OPPOSING, NEUTRAL_ADDITIVE
    epistemic_status: str        # INTERACTION_OBSERVED vs ADDITIVE_CONSISTENT
    metadata: Dict = field(default_factory=dict)


# =============================================================================
# 2. Primitive Operations
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_attention_clamping_intervention(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    base_logits: torch.Tensor,
    corrupted_toks: torch.Tensor,
    pos: int = -1
) -> Tuple[float, float, float, float, str]:
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())

    # Natural Intervention
    def patch_res_hook(act, hook):
        act[0, pos, :] += delta_contrib
        return act

    with torch.no_grad():
        natural_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", patch_res_hook)])

    natural_diff = float((natural_logits[0, pos, correct_id] - natural_logits[0, pos, incorrect_id]).item())
    natural_effect = natural_diff - base_diff

    # Clamped Intervention
    corrupted_pattern = corrupted_cache[f"blocks.{l_tgt}.attn.hook_pattern"][0, h_tgt, :, :].clone()

    def clamp_attention_pattern_hook(act, hook):
        act[0, h_tgt, :, :] = corrupted_pattern
        return act

    clamped_hooks = [
        (f"blocks.{l_tgt}.hook_resid_pre", patch_res_hook),
        (f"blocks.{l_tgt}.attn.hook_pattern", clamp_attention_pattern_hook)
    ]

    with torch.no_grad():
        clamped_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=clamped_hooks)

    clamped_diff = float((clamped_logits[0, pos, correct_id] - clamped_logits[0, pos, incorrect_id]).item())
    clamped_effect = clamped_diff - base_diff

    mediated_effect = natural_effect - clamped_effect
    abs_natural = abs(natural_effect)
    mediation_ratio = (mediated_effect / abs_natural) if abs_natural > 1e-6 else 0.0

    if abs(mediated_effect) < MEDIATION_EFFECT_FLOOR:
        direction_label = "NEUTRAL"
    elif np.sign(natural_effect) == np.sign(mediated_effect):
        direction_label = "AMPLIFYING" if abs(natural_effect) > abs(clamped_effect) else "SUPPRESSIVE"
    else:
        direction_label = "OPPOSING"

    return natural_effect, clamped_effect, mediated_effect, mediation_ratio, direction_label


# =============================================================================
# 3. Engines Execution Pipeline
# =============================================================================

class EpistemicEvidenceEngine:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()
        self.rng = np.random.default_rng(42)

    def generate_clean_null_distribution(
        self,
        target: Tuple[int, int],
        task_data: List,
        exclude_sources: Set[Tuple[int, int]],
        k_samples: int = DEFAULT_K_NULL_SAMPLES
    ) -> Tuple[float, float, Optional[float], str, List[float]]:
        l_tgt, _ = target
        possible_null_sources = [
            (l, h) for l in range(l_tgt) for h in range(12)
            if (l, h) not in exclude_sources
        ]

        if len(possible_null_sources) < k_samples:
            null_sources = possible_null_sources
        else:
            indices = self.rng.choice(len(possible_null_sources), k_samples, replace=False)
            null_sources = [possible_null_sources[idx] for idx in indices]

        null_deltas = []
        for src in null_sources:
            task_effects = []
            for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                correct_id = int(self.model.to_single_token(task["correct"]))
                incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[target[0], target[1]]

                def patch_v(act, hook):
                    act[0, -1, target[1], :] += delta_v
                    return act

                with torch.no_grad():
                    patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.attn.hook_v", patch_v)])

                p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                task_effects.append(p_diff - base_diff)
            null_deltas.append(float(np.mean(task_effects)))

        null_mean = float(np.mean(null_deltas))
        null_std = float(np.std(null_deltas))

        if null_std < 1e-6:
            variance_status = "NULL_VARIANCE_TOO_SMALL"
            null_z = None
        else:
            variance_status = "VALID"
            null_z = None

        return null_mean, null_std, null_z, variance_status, null_deltas

    def execute_pipeline(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        k_null_samples: int = DEFAULT_K_NULL_SAMPLES
    ) -> SelfCircuitGraph:

        graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])
        n_tasks = len(tasks)

        print(f"\n⚡️ [EPISTEMIC ENGINE] Running Pipeline for Regime '{regime_id}' across {n_tasks} task(s)")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, base_corrupted_logits, clean_logits))

        for j in range(len(sorted_nodes)):
            tgt = sorted_nodes[j]
            target_candidates = {sorted_nodes[i] for i in range(j) if sorted_nodes[i][0] < tgt[0]}

            for i in range(j):
                src = sorted_nodes[i]
                if src[0] >= tgt[0]:
                    continue

                evidence_ladder = EvidenceLadder()

                # Level 1
                raw_effects = []
                for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                    contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                    delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[tgt[0], tgt[1]]

                    def patch_v(act, hook):
                        act[0, -1, tgt[1], :] += delta_v
                        return act

                    with torch.no_grad():
                        patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{tgt[0]}.attn.hook_v", patch_v)])
                    p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                    raw_effects.append(p_diff - base_diff)

                avg_raw = float(np.mean(raw_effects))
                null_mean, null_std, _, var_status, null_deltas = self.generate_clean_null_distribution(
                    target=tgt, task_data=task_data, exclude_sources=target_candidates, k_samples=k_null_samples
                )

                if len(null_deltas) > 0:
                    empirical_p = float((np.sum(np.abs(null_deltas) >= abs(avg_raw)) + 1) / (len(null_deltas) + 1))
                else:
                    empirical_p = None

                computed_z = (avg_raw - null_mean) / null_std if var_status == "VALID" else None

                evidence_ladder.level1_target_value = InterventionEvidence(
                    intervention_type="target_value_intervention", source=src, target=tgt,
                    intervention_site=f"blocks.{tgt[0]}.attn.hook_v", raw_effect=avg_raw,
                    null_mean=null_mean, null_std=null_std, null_z=computed_z,
                    empirical_p=empirical_p, null_variance_status=var_status,
                    n_null_samples=k_null_samples, metadata={"n_tasks": n_tasks}
                )

                # Level 2
                l1_valid = (empirical_p is not None and empirical_p <= STATISTICAL_P_THRESHOLD) or (computed_z is not None and abs(computed_z) >= 3.0)

                if l1_valid and abs(avg_raw) >= RAW_INFLUENCE_FLOOR:
                    z_deltas, out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        contrib_corrupt = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                        delta_contrib = contrib_clean - contrib_corrupt

                        def patch_res(act, hook):
                            act[0, -1, :] += delta_contrib
                            return act

                        target_z_patched = [None]
                        def cap_z(act, hook):
                            target_z_patched[0] = act[0, -1, tgt[1], :].clone()
                            return act

                        hooks = [(f"blocks.{tgt[0]}.hook_resid_pre", patch_res), (f"blocks.{tgt[0]}.attn.hook_z", cap_z)]
                        with torch.no_grad():
                            patched_logits = self.model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        b_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
                        p_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

                        z_corrupt = corrupted_cache[f"blocks.{tgt[0]}.attn.hook_z"][0, -1, tgt[1], :]
                        z_deltas.append(float((target_z_patched[0] - z_corrupt).norm().item()))
                        out_deltas.append(p_diff - b_diff)

                    evidence_ladder.level2_residual_path = InterventionEvidence(
                        intervention_type="residual_pre_patch", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre",
                        target_delta=float(np.mean(z_deltas)), output_delta=float(np.mean(out_deltas)),
                        metadata={"n_tasks": n_tasks}
                    )

                    # Level 3
                    nat_effs, clamp_effs, med_effs, ratios, directions = [], [], [], [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        nat_e, clamp_e, med_e, ratio, dir_label = evaluate_attention_clamping_intervention(
                            self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks
                        )
                        nat_effs.append(nat_e)
                        clamp_effs.append(clamp_e)
                        med_effs.append(med_e)
                        ratios.append(ratio)
                        directions.append(dir_label)

                    avg_med = float(np.mean(med_effs))
                    avg_nat = float(np.mean(nat_effs))
                    avg_ratio = (avg_med / abs(avg_nat)) if abs(avg_nat) > 1e-6 else 0.0

                    if abs(avg_med) < MEDIATION_EFFECT_FLOOR:
                        overall_dir = "NEUTRAL"
                    elif np.sign(avg_nat) == np.sign(avg_med):
                        overall_dir = "AMPLIFYING" if abs(avg_nat) > abs(float(np.mean(clamp_effs))) else "SUPPRESSIVE"
                    else:
                        overall_dir = "OPPOSING"

                    evidence_ladder.level3_attention_effect = InterventionEvidence(
                        intervention_type="interventional_attention_clamping", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.attn.hook_pattern",
                        natural_effect=avg_nat,
                        clamped_effect=float(np.mean(clamp_effs)),
                        mediated_effect=avg_med,
                        mediation_ratio=avg_ratio,
                        direction_label=overall_dir,
                        metadata={"n_tasks": n_tasks}
                    )

                edge = EvidenceEdge(source=src, target=tgt, regime_id=regime_id, evidence=evidence_ladder)
                graph.add_edge(edge)

        return graph


class InteractionDiscoveryEngine:
    def __init__(self, engine: EpistemicEvidenceEngine, circuit_graph: SelfCircuitGraph):
        self.engine = engine
        self.graph = circuit_graph

    def evaluate_pair_interaction(
        self,
        source_a: Tuple[int, int],
        source_b: Tuple[int, int],
        target: Tuple[int, int],
        tasks: List[Dict]
    ) -> Order2InteractionEvidence:
        l_tgt, h_tgt = target
        l_a, h_a = source_a
        l_b, h_b = source_b

        effects_a, effects_b, effects_ab = [], [], []

        for task in tasks:
            clean_toks = self.engine.model.to_tokens(task["clean"])
            corrupted_toks = self.engine.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                _, clean_cache = self.engine.model.run_with_cache(clean_toks)
                base_logits, corrupted_cache = self.engine.model.run_with_cache(corrupted_toks)

            correct_id = int(self.engine.model.to_single_token(task["correct"]))
            incorrect_id = int(self.engine.model.to_single_token(task["incorrect"]))
            base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

            contrib_a_clean = compute_head_contribution(self.engine.model, clean_cache, l_a, h_a)
            contrib_a_corrupt = compute_head_contribution(self.engine.model, corrupted_cache, l_a, h_a)
            delta_contrib_a = contrib_a_clean - contrib_a_corrupt

            contrib_b_clean = compute_head_contribution(self.engine.model, clean_cache, l_b, h_b)
            contrib_b_corrupt = compute_head_contribution(self.engine.model, corrupted_cache, l_b, h_b)
            delta_contrib_b = contrib_b_clean - contrib_b_corrupt

            # 1. Delta_A
            def hook_a(act, hook):
                act[0, -1, :] += delta_contrib_a
                return act

            with torch.no_grad():
                logits_a = self.engine.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_a)])
            diff_a = float((logits_a[0, -1, correct_id] - logits_a[0, -1, incorrect_id]).item())
            effects_a.append(diff_a - base_diff)

            # 2. Delta_B
            def hook_b(act, hook):
                act[0, -1, :] += delta_contrib_b
                return act

            with torch.no_grad():
                logits_b = self.engine.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_b)])
            diff_b = float((logits_b[0, -1, correct_id] - logits_b[0, -1, incorrect_id]).item())
            effects_b.append(diff_b - base_diff)

            # 3. Delta_AB
            def hook_ab(act, hook):
                act[0, -1, :] += (delta_contrib_a + delta_contrib_b)
                return act

            with torch.no_grad():
                logits_ab = self.engine.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_ab)])
            diff_ab = float((logits_ab[0, -1, correct_id] - logits_ab[0, -1, incorrect_id]).item())
            effects_ab.append(diff_ab - base_diff)

        avg_a = float(np.mean(effects_a))
        avg_b = float(np.mean(effects_b))
        avg_ab = float(np.mean(effects_ab))

        additive_pred = avg_a + avg_b
        i_ab = avg_ab - additive_pred
        i_rel = float(abs(i_ab) / (abs(avg_ab) + 1e-6))

        if abs(i_ab) < 0.0100:
            i_type = "NEUTRAL_ADDITIVE"
            status = "ADDITIVE_CONSISTENT"
        else:
            status = "INTERACTION_OBSERVED"
            if np.sign(avg_ab) == np.sign(additive_pred):
                i_type = "SYNERGISTIC" if abs(avg_ab) > abs(additive_pred) else "CANCELLING"
            else:
                i_type = "OPPOSING"

        return Order2InteractionEvidence(
            source_a=source_a,
            source_b=source_b,
            target=target,
            regime_id=self.graph.regime_id,
            delta_a=avg_a,
            delta_b=avg_b,
            delta_ab=avg_ab,
            additive_prediction=additive_pred,
            interaction_term=i_ab,
            relative_interaction=i_rel,
            interaction_type=i_type,
            epistemic_status=status,
            metadata={"n_tasks": len(tasks), "k_null_resolution_limit": 1 / (DEFAULT_K_NULL_SAMPLES + 1)}
        )


# =============================================================================
# 4. Main Execution
# =============================================================================

if __name__ == "__main__":
    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    engine = EpistemicEvidenceEngine("gpt2")
    evidence_graph = engine.execute_pipeline(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        k_null_samples=DEFAULT_K_NULL_SAMPLES
    )

    interaction_engine = InteractionDiscoveryEngine(engine, evidence_graph)

    evidence = interaction_engine.evaluate_pair_interaction(
        source_a=(9, 9),
        source_b=(10, 7),
        target=(11, 10),
        tasks=IOI_TASKS
    )

    print("\n==================================================")
    print("🔬 MILESTONE 3.1: ORDER-2 INTERACTION DISCOVERY RESULTS")
    print("==================================================")
    print(f"Target: L{evidence.target[0]}H{evidence.target[1]} | Sources: L{evidence.source_a[0]}H{evidence.source_a[1]} & L{evidence.source_b[0]}H{evidence.source_b[1]}")
    print(f"Status: [{evidence.epistemic_status}] | Type: [{evidence.interaction_type}]")
    print(f"  ├── Delta_A (L9H9):                 {evidence.delta_a:+6.4f}")
    print(f"  ├── Delta_B (L10H7):                {evidence.delta_b:+6.4f}")
    print(f"  ├── Additive Prediction (A + B):    {evidence.additive_prediction:+6.4f}")
    print(f"  ├── Actual Joint Effect (Delta_AB): {evidence.delta_ab:+6.4f}")
    print(f"  ├── Interaction Term (I_AB):        {evidence.interaction_term:+6.4f}")
    print(f"  └── Relative Interaction (I_rel):   {evidence.relative_interaction * 100:5.1f}%")
    print("==================================================")
    print(f"Note: Evaluated on N={evidence.metadata['n_tasks']} tasks. Null-tail min resolution = {evidence.metadata['k_null_resolution_limit']:.4f}.")
    print("==================================================\n")

/tmp/ipykernel_974/1394597324.py:211: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [EPISTEMIC ENGINE] Running Pipeline for Regime 'R1_IOI' across 2 task(s)

🔬 MILESTONE 3.1: ORDER-2 INTERACTION DISCOVERY RESULTS
Target: L11H10 | Sources: L9H9 & L10H7
Status: [INTERACTION_OBSERVED] | Type: [SYNERGISTIC]
  ├── Delta_A (L9H9):                 +1.9751
  ├── Delta_B (L10H7):                -1.7838
  ├── Additive Prediction (A + B):    +0.1913
  ├── Actual Joint Effect (Delta_AB): +0.4956
  ├── Interaction Term (I_AB):        +0.3043
  └── Relative Interaction (I_rel):    61.4%
Note: Evaluated on N=2 tasks. Null-tail min resolution = 0.0154.



In [10]:
"""
milestone3_2_unseen_task_prediction.py
======================================
Milestone 3.2 - Generalization Benchmark for Discovered Interaction Term (I_AB).
- Train/Test Split on IOI Prompts:
    * Train Tasks: Discover Delta_A, Delta_B, and I_AB.
    * Test Tasks (Unseen): Evaluate Additive Model vs Interaction-Aware Model.
- Evaluates if adding learned I_AB reduces MAE on new contexts.
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer


def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def measure_intervention_effects(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    tasks: List[Dict]
) -> Tuple[float, float, float]:
    """محاسبه اثرات منفرد و ترکیبی مداخله روی یک مجموعه از مسائل."""
    l_tgt, h_tgt = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    effects_a, effects_b, effects_ab = [], [], []

    for task in tasks:
        clean_toks = model.to_tokens(task["clean"])
        corrupted_toks = model.to_tokens(task["corrupted"])

        with torch.no_grad():
            _, clean_cache = model.run_with_cache(clean_toks)
            base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

        correct_id = int(model.to_single_token(task["correct"]))
        incorrect_id = int(model.to_single_token(task["incorrect"]))
        base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

        contrib_a_clean = compute_head_contribution(model, clean_cache, l_a, h_a)
        contrib_a_corrupt = compute_head_contribution(model, corrupted_cache, l_a, h_a)
        delta_contrib_a = contrib_a_clean - contrib_a_corrupt

        contrib_b_clean = compute_head_contribution(model, clean_cache, l_b, h_b)
        contrib_b_corrupt = compute_head_contribution(model, corrupted_cache, l_b, h_b)
        delta_contrib_b = contrib_b_clean - contrib_b_corrupt

        # ۱. اثر A
        def hook_a(act, hook):
            act[0, -1, :] += delta_contrib_a
            return act

        with torch.no_grad():
            logits_a = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_a)])
        effects_a.append(float((logits_a[0, -1, correct_id] - logits_a[0, -1, incorrect_id]).item()) - base_diff)

        # ۲. اثر B
        def hook_b(act, hook):
            act[0, -1, :] += delta_contrib_b
            return act

        with torch.no_grad():
            logits_b = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_b)])
        effects_b.append(float((logits_b[0, -1, correct_id] - logits_b[0, -1, incorrect_id]).item()) - base_diff)

        # ۳. اثر ترکیبی A + B
        def hook_ab(act, hook):
            act[0, -1, :] += (delta_contrib_a + delta_contrib_b)
            return act

        with torch.no_grad():
            logits_ab = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_ab)])
        effects_ab.append(float((logits_ab[0, -1, correct_id] - logits_ab[0, -1, incorrect_id]).item()) - base_diff)

    return float(np.mean(effects_a)), float(np.mean(effects_b)), float(np.mean(effects_ab))


def run_unseen_task_benchmark(
    model_name: str = "gpt2",
    source_a: Tuple[int, int] = (9, 9),
    source_b: Tuple[int, int] = (10, 7),
    target: Tuple[int, int] = (11, 10)
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained(model_name, device=device)
    model.eval()

    # تفکیک داده‌های آموزش (کشف برهم‌کنش) و تست (مسائل ندیده)
    train_tasks = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    test_tasks = [
        {"clean": "Afterward, David and Sarah visited the library. David handed a book to", "corrupted": "Afterward, David and Sarah visited the library. Sarah handed a book to", "correct": " Sarah", "incorrect": " David"},
        {"clean": "Later, Charlie and Emma entered the room. Charlie brought a bag to", "corrupted": "Later, Charlie and Emma entered the room. Emma brought a bag to", "correct": " Emma", "incorrect": " Charlie"}
    ]

    print("\n⚡️ [MILESTONE 3.2] Executing Unseen Task Generalization Benchmark")
    print(f"Target: L{target[0]}H{target[1]} | Sources: L{source_a[0]}H{source_a[1]} & L{source_b[0]}H{source_b[1]}")

    # -------------------------------------------------------------------------
    # فاز ۱: آموزش و کشف ترم برهم‌کنش I_AB روی Train Tasks
    # -------------------------------------------------------------------------
    delta_a_train, delta_b_train, delta_ab_train = measure_intervention_effects(
        model, source_a, source_b, target, train_tasks
    )

    learned_i_ab = delta_ab_train - (delta_a_train + delta_b_train)

    print("\n1️⃣ DISCOVERY PHASE (Train Tasks):")
    print(f"   ├── Train Delta_A:             {delta_a_train:+6.4f}")
    print(f"   ├── Train Delta_B:             {delta_b_train:+6.4f}")
    print(f"   ├── Train Delta_AB (Actual):   {delta_ab_train:+6.4f}")
    print(f"   └── Learned Interaction (I_AB): {learned_i_ab:+6.4f}")

    # -------------------------------------------------------------------------
    # فاز ۲: ارزیابی پیش‌بینی روی Test Tasks (مسائل ندیده)
    # -------------------------------------------------------------------------
    delta_a_test, delta_b_test, actual_ab_test = measure_intervention_effects(
        model, source_a, source_b, target, test_tasks
    )

    # مدل ۱: پیش‌بینی جمع‌پذیر ساده (Additive Baseline)
    pred_additive_test = delta_a_test + delta_b_test
    mae_additive = abs(actual_ab_test - pred_additive_test)

    # مدل ۲: پیش‌بینی با اعمال ترم برهم‌کنش کشف‌شده (Interaction-Aware Model)
    pred_interaction_test = delta_a_test + delta_b_test + learned_i_ab
    mae_interaction = abs(actual_ab_test - pred_interaction_test)

    # محاسبه درصد بهبود خطا
    mae_reduction_percent = ((mae_additive - mae_interaction) / mae_additive) * 100

    print("\n2️⃣ EVALUATION PHASE (Unseen Test Tasks):")
    print(f"   ├── Test Delta_A (Single):              {delta_a_test:+6.4f}")
    print(f"   ├── Test Delta_B (Single):              {delta_b_test:+6.4f}")
    print(f"   ├── Ground Truth Joint Effect (Actual): {actual_ab_test:+6.4f}")
    print("   │")
    print(f"   ├── Additive Model Prediction:          {pred_additive_test:+6.4f}  |  MAE: {mae_additive:6.4f}")
    print(f"   └── Interaction Model Prediction:       {pred_interaction_test:+6.4f}  |  MAE: {mae_interaction:6.4f}")

    print("\n==================================================")
    print("📊 BENCHMARK EVALUATION RESULTS")
    print("==================================================")
    print(f"Additive Model MAE:          {mae_additive:6.4f}")
    print(f"Interaction-Aware Model MAE: {mae_interaction:6.4f}")
    print(f"Error Reduction:             {mae_reduction_percent:5.1f}%")
    print("==================================================\n")


if __name__ == "__main__":
    run_unseen_task_benchmark()

/tmp/ipykernel_974/2858472680.py:95: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(model_name, device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 3.2] Executing Unseen Task Generalization Benchmark
Target: L11H10 | Sources: L9H9 & L10H7

1️⃣ DISCOVERY PHASE (Train Tasks):
   ├── Train Delta_A:             +1.9751
   ├── Train Delta_B:             -1.7838
   ├── Train Delta_AB (Actual):   +0.4956
   └── Learned Interaction (I_AB): +0.3043

2️⃣ EVALUATION PHASE (Unseen Test Tasks):
   ├── Test Delta_A (Single):              +1.7695
   ├── Test Delta_B (Single):              -1.9890
   ├── Ground Truth Joint Effect (Actual): +0.3422
   │
   ├── Additive Model Prediction:          -0.2195  |  MAE: 0.5617
   └── Interaction Model Prediction:       +0.0848  |  MAE: 0.2574

📊 BENCHMARK EVALUATION RESULTS
Additive Model MAE:          0.5617
Interaction-Aware Model MAE: 0.2574
Error Reduction:              54.2%



In [11]:
"""
milestone3_2_benchmark.py
=========================
Milestone 3.2 - Unseen Task Generalization Benchmark (Refactored).

Features:
- Strict Train/Test Separation (Zero Leakage).
- Task-level metrics calculation & aggregation.
- Safe division for relative improvements.
- Honest epistemic metadata reporting (NO ungrounded claims).
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer


# =============================================================================
# 1. Data Schemas
# =============================================================================

@dataclass
class TrainTaskResult:
    task_index: int
    prompt_desc: str
    delta_a: float
    delta_b: float
    delta_ab: float
    i_ab: float


@dataclass
class TestTaskResult:
    task_index: int
    prompt_desc: str
    delta_a: float
    delta_b: float
    actual_delta_ab: float
    pred_additive: float
    err_additive: float
    pred_interaction: float
    err_interaction: float


# =============================================================================
# 2. Intervention Primitives
# =============================================================================

def compute_head_contribution(
    model: HookedTransformer,
    cache: Dict,
    layer: int,
    head: int,
    pos: int = -1
) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_single_task_effects(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    pos: int = -1
) -> Tuple[float, float, float]:
    """
    محاسبه اثرات Delta_A، Delta_B و Delta_AB برای یک Task به‌صورت ایزوله.
    """
    l_tgt, _ = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    with torch.no_grad():
        _, clean_cache = model.run_with_cache(clean_toks)
        base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())

    # Delta contributions
    contrib_a_clean = compute_head_contribution(model, clean_cache, l_a, h_a, pos)
    contrib_a_corrupt = compute_head_contribution(model, corrupted_cache, l_a, h_a, pos)
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contribution(model, clean_cache, l_b, h_b, pos)
    contrib_b_corrupt = compute_head_contribution(model, corrupted_cache, l_b, h_b, pos)
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    # 1. Delta_A
    def hook_a(act, hook):
        act[0, pos, :] += delta_a_vec
        return act

    with torch.no_grad():
        logits_a = model.run_with_hooks(
            corrupted_toks,
            fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_a)]
        )
    diff_a = float((logits_a[0, pos, correct_id] - logits_a[0, pos, incorrect_id]).item())
    delta_a = diff_a - base_diff

    # 2. Delta_B
    def hook_b(act, hook):
        act[0, pos, :] += delta_b_vec
        return act

    with torch.no_grad():
        logits_b = model.run_with_hooks(
            corrupted_toks,
            fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_b)]
        )
    diff_b = float((logits_b[0, pos, correct_id] - logits_b[0, pos, incorrect_id]).item())
    delta_b = diff_b - base_diff

    # 3. Delta_AB (Joint)
    def hook_ab(act, hook):
        act[0, pos, :] += (delta_a_vec + delta_b_vec)
        return act

    with torch.no_grad():
        logits_ab = model.run_with_hooks(
            corrupted_toks,
            fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_ab)]
        )
    diff_ab = float((logits_ab[0, pos, correct_id] - logits_ab[0, pos, incorrect_id]).item())
    delta_ab = diff_ab - base_diff

    return delta_a, delta_b, delta_ab


# =============================================================================
# 3. Benchmark Execution Pipeline
# =============================================================================

def run_unseen_task_benchmark(
    model_name: str = "gpt2",
    source_a: Tuple[int, int] = (9, 9),
    source_b: Tuple[int, int] = (10, 7),
    target: Tuple[int, int] = (11, 10),
    train_tasks: List[Dict] = None,
    test_tasks: List[Dict] = None
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained(model_name, device=device)
    model.eval()

    if train_tasks is None or test_tasks is None:
        raise ValueError("Both train_tasks and test_tasks must be provided.")

    n_train = len(train_tasks)
    n_test = len(test_tasks)

    print("\n==================================================")
    print("⚡️ [MILESTONE 3.2] BENCHMARK EXECUTION")
    print("==================================================")
    print(f"Target: L{target[0]}H{target[1]} | Sources: L{source_a[0]}H{source_a[1]} & L{source_b[0]}H{source_b[1]}")
    print(f"Dataset Scope: N_train = {n_train}, N_test = {n_test}")
    print("Metadata Status: [OUT_OF_SAMPLE_EVALUATION] [LIMITED_SAMPLE_RESULT]")

    # -------------------------------------------------------------------------
    # STEP 1: TRAIN / DISCOVERY PHASE
    # -------------------------------------------------------------------------
    train_results: List[TrainTaskResult] = []

    for idx, task in enumerate(train_tasks):
        d_a, d_b, d_ab = evaluate_single_task_effects(model, source_a, source_b, target, task)
        i_ab = d_ab - (d_a + d_b)

        train_results.append(
            TrainTaskResult(
                task_index=idx + 1,
                prompt_desc=task["clean"][:35] + "...",
                delta_a=d_a,
                delta_b=d_b,
                delta_ab=d_ab,
                i_ab=i_ab
            )
        )

    train_i_ab_vals = [r.i_ab for r in train_results]
    i_ab_train_mean = float(np.mean(train_i_ab_vals))
    i_ab_train_std = float(np.std(train_i_ab_vals))

    print("\n--------------------------------------------------")
    print("1️⃣ TRAIN PHASE (Discovery Results)")
    print("--------------------------------------------------")
    for r in train_results:
        print(f"  Task {r.task_index} [{r.prompt_desc}]")
        print(f"     Delta_A: {r.delta_a:+6.4f} | Delta_B: {r.delta_b:+6.4f} | Delta_AB: {r.delta_ab:+6.4f} | I_AB: {r.i_ab:+6.4f}")

    print(f"\n  ★ Learned Train I_AB Mean: {i_ab_train_mean:+6.4f} (Std: {i_ab_train_std:6.4f})")

    # -------------------------------------------------------------------------
    # STEP 2: TEST / EVALUATION PHASE
    # -------------------------------------------------------------------------
    test_results: List[TestTaskResult] = []

    for idx, task in enumerate(test_tasks):
        d_a, d_b, actual_ab = evaluate_single_task_effects(model, source_a, source_b, target, task)

        pred_additive = d_a + d_b
        err_additive = abs(actual_ab - pred_additive)

        pred_interaction = d_a + d_b + i_ab_train_mean
        err_interaction = abs(actual_ab - pred_interaction)

        test_results.append(
            TestTaskResult(
                task_index=idx + 1,
                prompt_desc=task["clean"][:35] + "...",
                delta_a=d_a,
                delta_b=d_b,
                actual_delta_ab=actual_ab,
                pred_additive=pred_additive,
                err_additive=err_additive,
                pred_interaction=pred_interaction,
                err_interaction=err_interaction
            )
        )

    print("\n--------------------------------------------------")
    print("2️⃣ TEST PHASE (Unseen Tasks Per-Task Results)")
    print("--------------------------------------------------")
    for r in test_results:
        print(f"  Task {r.task_index} [{r.prompt_desc}]")
        print(f"     Actual Delta_AB: {r.actual_delta_ab:+6.4f}")
        print(f"     ├── Additive Pred:    {r.pred_additive:+6.4f} | Abs Error: {r.err_additive:6.4f}")
        print(f"     └── Interaction Pred: {r.pred_interaction:+6.4f} | Abs Error: {r.err_interaction:6.4f}")

    # -------------------------------------------------------------------------
    # STEP 3: AGGREGATE METRICS & REPORTING
    # -------------------------------------------------------------------------
    mae_additive = float(np.mean([r.err_additive for r in test_results]))
    mae_interaction = float(np.mean([r.err_interaction for r in test_results]))

    abs_improvement = mae_additive - mae_interaction

    if mae_additive > 1e-8:
        rel_improvement_pct = (abs_improvement / mae_additive) * 100.0
    else:
        rel_improvement_pct = 0.0

    print("\n==================================================")
    print("📊 BENCHMARK SUMMARY REPORT")
    print("==================================================")
    print(f"N_train: {n_train} | N_test: {n_test}")
    print(f"Train I_AB Mean: {i_ab_train_mean:+6.4f} | Std: {i_ab_train_std:6.4f}")
    print("--------------------------------------------------")
    print(f"Test MAE - Additive:          {mae_additive:6.4f}")
    print(f"Test MAE - Interaction:       {mae_interaction:6.4f}")
    print(f"Absolute MAE Improvement:     {abs_improvement:+6.4f}")
    print(f"Relative MAE Improvement (%): {rel_improvement_pct:5.1f}%")
    print("==================================================")
    print("EPISTEMIC METADATA:")
    print("  - Observation: [OBSERVED]")
    print(f"  - Performance: [{'PREDICTIVE_IMPROVEMENT' if abs_improvement > 0 else 'NO_IMPROVEMENT'}]")
    print("  - Note: This benchmark uses N_train = {} and N_test = {}.".format(n_train, n_test))
    print("  - Constraint: Results are task-level observations and are NOT claims about GPT-2 in general.")
    print("==================================================\n")


# =============================================================================
# 4. Execution Entry Point
# =============================================================================

if __name__ == "__main__":
    TRAIN_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    TEST_TASKS = [
        {"clean": "Afterward, David and Sarah visited the library. David handed a book to", "corrupted": "Afterward, David and Sarah visited the library. Sarah handed a book to", "correct": " Sarah", "incorrect": " David"},
        {"clean": "Later, Charlie and Emma entered the room. Charlie brought a bag to", "corrupted": "Later, Charlie and Emma entered the room. Emma brought a bag to", "correct": " Emma", "incorrect": " Charlie"}
    ]

    run_unseen_task_benchmark(
        model_name="gpt2",
        source_a=(9, 9),
        source_b=(10, 7),
        target=(11, 10),
        train_tasks=TRAIN_TASKS,
        test_tasks=TEST_TASKS
    )

/tmp/ipykernel_974/3967798860.py:153: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(model_name, device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 3.2] BENCHMARK EXECUTION
Target: L11H10 | Sources: L9H9 & L10H7
Dataset Scope: N_train = 2, N_test = 2
Metadata Status: [OUT_OF_SAMPLE_EVALUATION] [LIMITED_SAMPLE_RESULT]

--------------------------------------------------
1️⃣ TRAIN PHASE (Discovery Results)
--------------------------------------------------
  Task 1 [Then, John and Mary went to the sto...]
     Delta_A: +1.6236 | Delta_B: -1.2840 | Delta_AB: +0.1005 | I_AB: -0.2391
  Task 2 [Then, Alice and Bob went to the par...]
     Delta_A: +2.3267 | Delta_B: -2.2836 | Delta_AB: +0.8908 | I_AB: +0.8477

  ★ Learned Train I_AB Mean: +0.3043 (Std: 0.5434)

--------------------------------------------------
2️⃣ TEST PHASE (Unseen Tasks Per-Task Results)
--------------------------------------------------
  Task 1 [Afterward, David and Sarah visited ...]
     Actual Delta_AB: +0.3426
     ├── Additive Pred:    -0.7609 | Abs Error: 1.1034
     └── Interaction Pred: -0.45

In [13]:
"""
milestone3_2_loto_controls_standalone.py
=========================================
Standalone implementation of LOTO Cross-Validation & Null Controls.
Does not require importing from external .py files.
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer


# =============================================================================
# Helper Functions & Intervention Primitives
# =============================================================================

def compute_head_contribution(
    model: HookedTransformer,
    cache: Dict,
    layer: int,
    head: int,
    pos: int = -1
) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_single_task_effects(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    pos: int = -1
) -> Tuple[float, float, float]:
    l_tgt, _ = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    with torch.no_grad():
        _, clean_cache = model.run_with_cache(clean_toks)
        base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())

    # Contributions
    contrib_a_clean = compute_head_contribution(model, clean_cache, l_a, h_a, pos)
    contrib_a_corrupt = compute_head_contribution(model, corrupted_cache, l_a, h_a, pos)
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contribution(model, clean_cache, l_b, h_b, pos)
    contrib_b_corrupt = compute_head_contribution(model, corrupted_cache, l_b, h_b, pos)
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    # 1. Delta_A
    def hook_a(act, hook):
        act[0, pos, :] += delta_a_vec
        return act

    with torch.no_grad():
        logits_a = model.run_with_hooks(
            corrupted_toks,
            fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_a)]
        )
    diff_a = float((logits_a[0, pos, correct_id] - logits_a[0, pos, incorrect_id]).item())
    delta_a = diff_a - base_diff

    # 2. Delta_B
    def hook_b(act, hook):
        act[0, pos, :] += delta_b_vec
        return act

    with torch.no_grad():
        logits_b = model.run_with_hooks(
            corrupted_toks,
            fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_b)]
        )
    diff_b = float((logits_b[0, pos, correct_id] - logits_b[0, pos, incorrect_id]).item())
    delta_b = diff_b - base_diff

    # 3. Delta_AB
    def hook_ab(act, hook):
        act[0, pos, :] += (delta_a_vec + delta_b_vec)
        return act

    with torch.no_grad():
        logits_ab = model.run_with_hooks(
            corrupted_toks,
            fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_ab)]
        )
    diff_ab = float((logits_ab[0, pos, correct_id] - logits_ab[0, pos, incorrect_id]).item())
    delta_ab = diff_ab - base_diff

    return delta_a, delta_b, delta_ab


# =============================================================================
# LOTO Engine Execution
# =============================================================================

def run_loto_null_controls(
    model_name: str = "gpt2",
    source_a: Tuple[int, int] = (9, 9),
    source_b: Tuple[int, int] = (10, 7),
    target: Tuple[int, int] = (11, 10),
    all_tasks: List[Dict] = None
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained(model_name, device=device)
    model.eval()

    n_tasks = len(all_tasks)
    print(f"\n⚡️ Running Leave-One-Task-Out (LOTO) across N = {n_tasks} tasks")

    # 1. Extract single task data
    task_data = []
    for idx, task in enumerate(all_tasks):
        d_a, d_b, d_ab = evaluate_single_task_effects(model, source_a, source_b, target, task)
        i_ab = d_ab - (d_a + d_b)
        task_data.append({
            "id": idx + 1,
            "desc": task["clean"][:30] + "...",
            "d_a": d_a,
            "d_b": d_b,
            "d_ab": d_ab,
            "i_ab": i_ab
        })

    # 2. Leave-One-Task-Out Loop
    errs_additive = []
    errs_loto_interaction = []
    errs_opposed_control = []
    errs_perturbed_control = []

    rng = np.random.default_rng(42)

    print("\n--------------------------------------------------")
    print("🔄 LOTO CROSS-VALIDATION RESULTS PER TASK")
    print("--------------------------------------------------")

    for i in range(n_tasks):
        test_item = task_data[i]
        train_items = [task_data[j] for j in range(n_tasks) if j != i]

        train_i_vals = [item["i_ab"] for item in train_items]
        learned_i_loto = float(np.mean(train_i_vals))

        d_a = test_item["d_a"]
        d_b = test_item["d_b"]
        actual = test_item["d_ab"]

        pred_add = d_a + d_b
        pred_loto = d_a + d_b + learned_i_loto
        pred_opposed = d_a + d_b - learned_i_loto

        noise = float(rng.normal(0, np.std(train_i_vals) + 1e-5))
        pred_perturbed = d_a + d_b + (learned_i_loto + noise)

        e_add = abs(actual - pred_add)
        e_loto = abs(actual - pred_loto)
        e_opp = abs(actual - pred_opposed)
        e_pert = abs(actual - pred_perturbed)

        errs_additive.append(e_add)
        errs_loto_interaction.append(e_loto)
        errs_opposed_control.append(e_opp)
        errs_perturbed_control.append(e_pert)

        print(f"Task {test_item['id']} [{test_item['desc']}]")
        print(f"  Task-specific I_AB: {test_item['i_ab']:+6.4f} | Learned LOTO I_AB: {learned_i_loto:+6.4f}")
        print(f"  ├── Additive Error:   {e_add:6.4f}")
        print(f"  ├── LOTO Inter Error: {e_loto:6.4f}")
        print(f"  └── Opposed Error:    {e_opp:6.4f}")

    # 3. Aggregation
    mae_add = float(np.mean(errs_additive))
    mae_loto = float(np.mean(errs_loto_interaction))
    mae_opp = float(np.mean(errs_opposed_control))
    mae_pert = float(np.mean(errs_perturbed_control))

    print("\n==================================================")
    print("📊 LOTO & NULL CONTROLS BENCHMARK SUMMARY")
    print("==================================================")
    print(f"1. Additive Baseline (I = 0):        MAE = {mae_add:6.4f}")
    print(f"2. LOTO Interaction Model:          MAE = {mae_loto:6.4f}")
    print(f"3. Opposed Control (I = -I_train):   MAE = {mae_opp:6.4f}")
    print(f"4. Random Perturbed Control:         MAE = {mae_pert:6.4f}")
    print("==================================================")
    print("METADATA:")
    print("  - Constraint: Evaluated via N-fold LOTO cross-validation.")
    print("  - Epistemic Status: [CONTEXT_DEPENDENCY_CHECK]")
    print("==================================================\n")


if __name__ == "__main__":
    ALL_IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"},
        {"clean": "Afterward, David and Sarah visited the library. David handed a book to", "corrupted": "Afterward, David and Sarah visited the library. Sarah handed a book to", "correct": " Sarah", "incorrect": " David"},
        {"clean": "Later, Charlie and Emma entered the room. Charlie brought a bag to", "corrupted": "Later, Charlie and Emma entered the room. Emma brought a bag to", "correct": " Emma", "incorrect": " Charlie"}
    ]

    run_loto_null_controls(all_tasks=ALL_IOI_TASKS)

/tmp/ipykernel_974/2401383775.py:117: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(model_name, device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ Running Leave-One-Task-Out (LOTO) across N = 4 tasks

--------------------------------------------------
🔄 LOTO CROSS-VALIDATION RESULTS PER TASK
--------------------------------------------------
Task 1 [Then, John and Mary went to th...]
  Task-specific I_AB: -0.2391 | Learned LOTO I_AB: +0.6571
  ├── Additive Error:   0.2391
  ├── LOTO Inter Error: 0.8962
  └── Opposed Error:    0.4179
Task 2 [Then, Alice and Bob went to th...]
  Task-specific I_AB: +0.8477 | Learned LOTO I_AB: +0.2948
  ├── Additive Error:   0.8477
  ├── LOTO Inter Error: 0.5529
  └── Opposed Error:    1.1425
Task 3 [Afterward, David and Sarah vis...]
  Task-specific I_AB: +1.1034 | Learned LOTO I_AB: +0.2096
  ├── Additive Error:   1.1034
  ├── LOTO Inter Error: 0.8939
  └── Opposed Error:    1.3130
Task 4 [Later, Charlie and Emma entere...]
  Task-specific I_AB: +0.0201 | Learned LOTO I_AB: +0.5707
  ├── Additive Error:   0.0201
  ├── LOTO Inter Error: 0.550

In [15]:
"""
milestone3_3_standalone_profiling.py
====================================
Milestone 3.3 - Integrated IOI Interaction Profiling (N=20).
Fixes tokenization assertion errors by dynamically verifying single-token names.
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer


# =============================================================================
# 1. Data Schema & Helper Functions
# =============================================================================

@dataclass
class PromptContextProfile:
    task_id: int
    prompt_text: str
    delta_a: float
    delta_b: float
    delta_ab: float
    i_ab: float
    clean_logit_diff: float
    resid_pre_norm: float


def generate_ioi_dataset_20(model: HookedTransformer) -> List[Dict]:
    """تولید ۲۰ مسئله IOI با فیلتر هوشمند اسامی تک‌توکنی در توکنایزر GPT-2."""
    candidate_pairs = [
        ("John", "Mary"), ("Alice", "Bob"), ("David", "Sarah"), ("Charlie", "Emma"),
        ("Paul", "Kate"), ("Mark", "Jane"), ("Luke", "Anne"), ("Tom", "Lucy"),
        ("Jack", "Amy"), ("Dan", "Eve"), ("Ben", "Grace"), ("Sam", "Mia"),
        ("Max", "Lia"), ("Leo", "Zoe"), ("Ray", "Joy"), ("Alex", "Kim"),
        ("Cole", "Fay"), ("Dean", "May"), ("Grant", "Hope"), ("Frank", "Ruth"),
        ("George", "Claire"), ("Henry", "Laura"), ("Peter", "Rachel")
    ]

    # فیلتر دقیق اسامی که حتماً ۱ توکن هستند
    valid_pairs = []
    for n1, n2 in candidate_pairs:
        tok_1 = model.to_tokens(f" {n1}", prepend_bos=False).squeeze()
        tok_2 = model.to_tokens(f" {n2}", prepend_bos=False).squeeze()
        if tok_1.ndim == 0 and tok_2.ndim == 0:
            valid_pairs.append((n1, n2))

    if len(valid_pairs) < 20:
        raise ValueError(f"فقط {len(valid_pairs)} جفت اسم تک‌توکنی یافت شد. لطفاً اسامی بیشتری اضافه کنید.")

    objects = ["drink", "toy", "book", "bag", "pen", "key", "apple", "letter", "box", "gift"]
    verbs = ["gave", "handed", "passed", "brought", "sent"]

    tasks = []
    for idx in range(20):
        n1, n2 = valid_pairs[idx]
        obj = objects[idx % len(objects)]
        verb = verbs[idx % len(verbs)]
        clean = f"Then, {n1} and {n2} went to the store. {n1} {verb} a {obj} to"
        corrupted = f"Then, {n1} and {n2} went to the store. {n2} {verb} a {obj} to"
        tasks.append({
            "clean": clean,
            "corrupted": corrupted,
            "correct": f" {n2}",
            "incorrect": f" {n1}"
        })
    return tasks


def compute_head_contrib(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


# =============================================================================
# 2. Execution Engine
# =============================================================================

def profile_single_task(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    task_id: int,
    pos: int = -1
) -> PromptContextProfile:
    l_tgt, _ = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    with torch.no_grad():
        clean_logits, clean_cache = model.run_with_cache(clean_toks)
        base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())
    clean_diff = float((clean_logits[0, pos, correct_id] - clean_logits[0, pos, incorrect_id]).item())
    resid_norm = float(corrupted_cache[f"blocks.{l_tgt}.hook_resid_pre"][0, pos, :].norm().item())

    # Delta contributions
    contrib_a_clean = compute_head_contrib(model, clean_cache, l_a, h_a, pos)
    contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, l_a, h_a, pos)
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contrib(model, clean_cache, l_b, h_b, pos)
    contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, l_b, h_b, pos)
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    # Interventions
    def hook_a(act, hook):
        act[0, pos, :] += delta_a_vec
        return act

    def hook_b(act, hook):
        act[0, pos, :] += delta_b_vec
        return act

    def hook_ab(act, hook):
        act[0, pos, :] += (delta_a_vec + delta_b_vec)
        return act

    with torch.no_grad():
        out_a = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_a)])
        out_b = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_b)])
        out_ab = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_ab)])

    delta_a = float((out_a[0, pos, correct_id] - out_a[0, pos, incorrect_id]).item()) - base_diff
    delta_b = float((out_b[0, pos, correct_id] - out_b[0, pos, incorrect_id]).item()) - base_diff
    delta_ab = float((out_ab[0, pos, correct_id] - out_ab[0, pos, incorrect_id]).item()) - base_diff

    i_ab = delta_ab - (delta_a + delta_b)

    return PromptContextProfile(
        task_id=task_id,
        prompt_text=task["clean"][:35] + "...",
        delta_a=delta_a,
        delta_b=delta_b,
        delta_ab=delta_ab,
        i_ab=i_ab,
        clean_logit_diff=clean_diff,
        resid_pre_norm=resid_norm
    )


def run_milestone3_3_profiling(
    model_name: str = "gpt2",
    source_a: Tuple[int, int] = (9, 9),
    source_b: Tuple[int, int] = (10, 7),
    target: Tuple[int, int] = (11, 10)
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained(model_name, device=device)
    model.eval()

    tasks = generate_ioi_dataset_20(model)
    profiles: List[PromptContextProfile] = []

    print(f"\n⚡️ [MILESTONE 3.3] Profiling Interaction Distribution across N={len(tasks)} IOI Prompts")
    print(f"Target: L{target[0]}H{target[1]} | Sources: L{source_a[0]}H{source_a[1]} & L{source_b[0]}H{source_b[1]}\n")

    for idx, task in enumerate(tasks):
        p = profile_single_task(model, source_a, source_b, target, task, idx + 1)
        profiles.append(p)
        print(f"Task {p.task_id:02d} | I_AB: {p.i_ab:+6.4f} | ResidNorm: {p.resid_pre_norm:6.2f} | CleanDiff: {p.clean_logit_diff:+5.2f}")

    i_ab_vals = np.array([p.i_ab for p in profiles])
    pos_count = int(np.sum(i_ab_vals > 0))
    neg_count = int(np.sum(i_ab_vals < 0))

    print("\n==================================================")
    print("📊 MILESTONE 3.3 PROFILING SUMMARY (N=20)")
    print("==================================================")
    print(f"Mean I_AB:              {np.mean(i_ab_vals):+6.4f}")
    print(f"Std I_AB:               {np.std(i_ab_vals):6.4f}")
    print(f"Min / Max I_AB:         {np.min(i_ab_vals):+6.4f} / {np.max(i_ab_vals):+6.4f}")
    print(f"Median I_AB:            {np.median(i_ab_vals):+6.4f}")
    print(f"Positive I_AB Ratio:    {pos_count}/20 ({pos_count/20*100:.1f}%) [Synergistic/Amplifying]")
    print(f"Negative I_AB Ratio:    {neg_count}/20 ({neg_count/20*100:.1f}%) [Suppressive/Cancelling]")
    print("==================================================")
    print("Context Correlation Candidates:")
    resid_norms = np.array([p.resid_pre_norm for p in profiles])
    clean_diffs = np.array([p.clean_logit_diff for p in profiles])
    print(f"Corr(I_AB, ResidNorm):  {np.corrcoef(i_ab_vals, resid_norms)[0, 1]:+6.4f}")
    print(f"Corr(I_AB, CleanDiff):  {np.corrcoef(i_ab_vals, clean_diffs)[0, 1]:+6.4f}")
    print("==================================================\n")


if __name__ == "__main__":
    run_milestone3_3_profiling()

/tmp/ipykernel_974/4171760213.py:161: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(model_name, device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 3.3] Profiling Interaction Distribution across N=20 IOI Prompts
Target: L11H10 | Sources: L9H9 & L10H7

Task 01 | I_AB: -0.2391 | ResidNorm: 240.63 | CleanDiff: +2.32
Task 02 | I_AB: +0.8165 | ResidNorm: 258.92 | CleanDiff: +2.20
Task 03 | I_AB: +1.1706 | ResidNorm: 237.65 | CleanDiff: +3.00
Task 04 | I_AB: +0.0369 | ResidNorm: 213.93 | CleanDiff: +2.17
Task 05 | I_AB: +1.3458 | ResidNorm: 239.25 | CleanDiff: +3.61
Task 06 | I_AB: +0.9168 | ResidNorm: 253.96 | CleanDiff: +2.90
Task 07 | I_AB: +0.8585 | ResidNorm: 246.48 | CleanDiff: +4.14
Task 08 | I_AB: +1.4928 | ResidNorm: 255.36 | CleanDiff: +4.61
Task 09 | I_AB: +0.0130 | ResidNorm: 215.13 | CleanDiff: +3.37
Task 10 | I_AB: +2.8096 | ResidNorm: 236.92 | CleanDiff: +5.57
Task 11 | I_AB: +2.3735 | ResidNorm: 225.64 | CleanDiff: +4.45
Task 12 | I_AB: +2.1460 | ResidNorm: 263.92 | CleanDiff: +4.57
Task 13 | I_AB: +1.8809 | ResidNorm: 235.94 | CleanDiff: +6.33
Task 14 | 

In [16]:
"""
milestone3_4_context_conditional_model.py
=========================================
Milestone 3.4 - Context-Conditioned Interaction Prediction (N=20, 5-Fold CV).

Feature Vector C:
  1. Clean Logit Difference (CleanDiff)
  2. Residual Stream Pre-Norm (ResidNorm)
  3. Norm of Delta_A vector (||Delta_A||)
  4. Norm of Delta_B vector (||Delta_B||)
  5. Cosine Similarity between Delta_A and Delta_B (cos(Delta_A, Delta_B))

Evaluated Models via 5-Fold Cross-Validation:
  - Model 0: Additive Baseline (I_hat = 0)
  - Model 1: Global Mean Baseline (I_hat = mean(I_train))
  - Model 2: Linear Context Model (I_hat = w^T * C + b)
  - Model 3: Shuffled-Label Control (Linear model fit on randomized train targets)
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold


@dataclass
class ContextFeatureProfile:
    task_id: int
    prompt_text: str
    delta_a_scalar: float
    delta_b_scalar: float
    delta_ab_scalar: float
    i_ab_actual: float

    # Context Features (C)
    clean_diff: float
    resid_norm: float
    norm_delta_a: float
    norm_delta_b: float
    cos_sim_delta: float

    def get_feature_vector(self) -> np.ndarray:
        return np.array([
            self.clean_diff,
            self.resid_norm,
            self.norm_delta_a,
            self.norm_delta_b,
            self.cos_sim_delta
        ], dtype=np.float64)


def generate_ioi_dataset_20(model: HookedTransformer) -> List[Dict]:
    """تولید ۲۰ مسئله IOI با فیلتر اسامی تک‌توکنی در توکنایزر GPT-2."""
    candidate_pairs = [
        ("John", "Mary"), ("Alice", "Bob"), ("David", "Sarah"), ("Charlie", "Emma"),
        ("Paul", "Kate"), ("Mark", "Jane"), ("Luke", "Anne"), ("Tom", "Lucy"),
        ("Jack", "Amy"), ("Dan", "Eve"), ("Ben", "Grace"), ("Sam", "Mia"),
        ("Max", "Lia"), ("Leo", "Zoe"), ("Ray", "Joy"), ("Alex", "Kim"),
        ("Cole", "Fay"), ("Dean", "May"), ("Grant", "Hope"), ("Frank", "Ruth")
    ]

    valid_pairs = []
    for n1, n2 in candidate_pairs:
        tok_1 = model.to_tokens(f" {n1}", prepend_bos=False).squeeze()
        tok_2 = model.to_tokens(f" {n2}", prepend_bos=False).squeeze()
        if tok_1.ndim == 0 and tok_2.ndim == 0:
            valid_pairs.append((n1, n2))

    objects = ["drink", "toy", "book", "bag", "pen", "key", "apple", "letter", "box", "gift"]
    verbs = ["gave", "handed", "passed", "brought", "sent"]

    tasks = []
    for idx in range(min(20, len(valid_pairs))):
        n1, n2 = valid_pairs[idx]
        obj = objects[idx % len(objects)]
        verb = verbs[idx % len(verbs)]
        clean = f"Then, {n1} and {n2} went to the store. {n1} {verb} a {obj} to"
        corrupted = f"Then, {n1} and {n2} went to the store. {n2} {verb} a {obj} to"
        tasks.append({
            "clean": clean,
            "corrupted": corrupted,
            "correct": f" {n2}",
            "incorrect": f" {n1}"
        })
    return tasks


def compute_head_contrib(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def extract_task_profile(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    task_id: int,
    pos: int = -1
) -> ContextFeatureProfile:
    l_tgt, _ = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    with torch.no_grad():
        clean_logits, clean_cache = model.run_with_cache(clean_toks)
        base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())
    clean_diff = float((clean_logits[0, pos, correct_id] - clean_logits[0, pos, incorrect_id]).item())
    resid_norm = float(corrupted_cache[f"blocks.{l_tgt}.hook_resid_pre"][0, pos, :].norm().item())

    # استخراج بردارهای Residual
    contrib_a_clean = compute_head_contrib(model, clean_cache, l_a, h_a, pos)
    contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, l_a, h_a, pos)
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contrib(model, clean_cache, l_b, h_b, pos)
    contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, l_b, h_b, pos)
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    # ویژگی‌های هندسی بردارها
    norm_a = float(delta_a_vec.norm().item())
    norm_b = float(delta_b_vec.norm().item())

    if norm_a > 1e-6 and norm_b > 1e-6:
        cos_sim = float(torch.dot(delta_a_vec, delta_b_vec) / (norm_a * norm_b))
    else:
        cos_sim = 0.0

    # مداخلات
    def hook_a(act, hook):
        act[0, pos, :] += delta_a_vec
        return act

    def hook_b(act, hook):
        act[0, pos, :] += delta_b_vec
        return act

    def hook_ab(act, hook):
        act[0, pos, :] += (delta_a_vec + delta_b_vec)
        return act

    with torch.no_grad():
        out_a = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_a)])
        out_b = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_b)])
        out_ab = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_ab)])

    delta_a_s = float((out_a[0, pos, correct_id] - out_a[0, pos, incorrect_id]).item()) - base_diff
    delta_b_s = float((out_b[0, pos, correct_id] - out_b[0, pos, incorrect_id]).item()) - base_diff
    delta_ab_s = float((out_ab[0, pos, correct_id] - out_ab[0, pos, incorrect_id]).item()) - base_diff

    i_ab = delta_ab_s - (delta_a_s + delta_b_s)

    return ContextFeatureProfile(
        task_id=task_id,
        prompt_text=task["clean"][:30] + "...",
        delta_a_scalar=delta_a_s,
        delta_b_scalar=delta_b_s,
        delta_ab_scalar=delta_ab_s,
        i_ab_actual=i_ab,
        clean_diff=clean_diff,
        resid_norm=resid_norm,
        norm_delta_a=norm_a,
        norm_delta_b=norm_b,
        cos_sim_delta=cos_sim
    )


def run_milestone3_4_benchmark(
    model_name: str = "gpt2",
    source_a: Tuple[int, int] = (9, 9),
    source_b: Tuple[int, int] = (10, 7),
    target: Tuple[int, int] = (11, 10),
    n_splits: int = 5
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained(model_name, device=device)
    model.eval()

    tasks = generate_ioi_dataset_20(model)
    profiles: List[ContextFeatureProfile] = []

    print(f"\n⚡️ [MILESTONE 3.4] Profiling & Extracting Vector Features for N={len(tasks)} Tasks...")

    for idx, task in enumerate(tasks):
        p = extract_task_profile(model, source_a, source_b, target, task, idx + 1)
        profiles.append(p)

    X = np.array([p.get_feature_vector() for p in profiles])
    y_actual = np.array([p.i_ab_actual for p in profiles])
    y_delta_ab = np.array([p.delta_ab_scalar for p in profiles])
    y_additive_pred = np.array([p.delta_a_scalar + p.delta_b_scalar for p in profiles])

    # -------------------------------------------------------------------------
    # 5-Fold Cross Validation Evaluation
    # -------------------------------------------------------------------------
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rng = np.random.default_rng(42)

    errs_additive = []
    errs_global_mean = []
    errs_context_linear = []
    errs_shuffled_control = []

    sign_correct_add = 0
    sign_correct_context = 0

    print(f"\n🔄 Executing {n_splits}-Fold Cross Validation (Context-Conditioned Models)...")

    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_actual[train_idx], y_actual[test_idx]
        ab_test_actual = y_delta_ab[test_idx]
        add_pred_test = y_additive_pred[test_idx]

        # 1. Scaler
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # 2. Global Mean Model
        global_mean_i = float(np.mean(y_train))

        # 3. Linear Context Model
        reg = LinearRegression()
        reg.fit(X_train_scaled, y_train)
        pred_i_context = reg.predict(X_test_scaled)

        # 4. Shuffled Control Model
        y_train_shuffled = y_train.copy()
        rng.shuffle(y_train_shuffled)
        reg_shuffled = LinearRegression()
        reg_shuffled.fit(X_train_scaled, y_train_shuffled)
        pred_i_shuffled = reg_shuffled.predict(X_test_scaled)

        # Calculate Joint Intervention Predictions
        pred_ab_additive = add_pred_test
        pred_ab_global_mean = add_pred_test + global_mean_i
        pred_ab_context = add_pred_test + pred_i_context
        pred_ab_shuffled = add_pred_test + pred_i_shuffled

        # Per-Task MAE on Test Fold
        errs_additive.extend(np.abs(ab_test_actual - pred_ab_additive))
        errs_global_mean.extend(np.abs(ab_test_actual - pred_ab_global_mean))
        errs_context_linear.extend(np.abs(ab_test_actual - pred_ab_context))
        errs_shuffled_control.extend(np.abs(ab_test_actual - pred_ab_shuffled))

        sign_correct_add += np.sum(np.sign(pred_ab_additive) == np.sign(ab_test_actual))
        sign_correct_context += np.sum(np.sign(pred_ab_context) == np.sign(ab_test_actual))

    mae_add = float(np.mean(errs_additive))
    mae_mean = float(np.mean(errs_global_mean))
    mae_context = float(np.mean(errs_context_linear))
    mae_shuffled = float(np.mean(errs_shuffled_control))

    imp_context_vs_add = ((mae_add - mae_context) / mae_add) * 100
    imp_context_vs_mean = ((mae_mean - mae_context) / mae_mean) * 100

    print("\n==================================================")
    print(f"📊 MILESTONE 3.4: 5-FOLD CV BENCHMARK REPORT (N={len(tasks)})")
    print("==================================================")
    print(f"1. Additive Baseline (I_hat = 0):        MAE = {mae_add:6.4f}")
    print(f"2. Global Mean Baseline (I_hat = I_bar): MAE = {mae_mean:6.4f}")
    print(f"3. Linear Context Model (I_hat = f(C)):  MAE = {mae_context:6.4f}")
    print(f"4. Shuffled-Label Control:               MAE = {mae_shuffled:6.4f}")
    print("--------------------------------------------------")
    print(f"Improvement over Additive Baseline:      {imp_context_vs_add:+5.1f}%")
    print(f"Improvement over Global Mean Baseline:   {imp_context_vs_mean:+5.1f}%")
    print(f"Sign Accuracy (Additive):                {sign_correct_add}/{len(tasks)} ({sign_correct_add/len(tasks)*100:.1f}%)")
    print(f"Sign Accuracy (Context Model):           {sign_correct_context}/{len(tasks)} ({sign_correct_context/len(tasks)*100:.1f}%)")
    print("==================================================")
    print("EPISTEMIC METADATA:")
    print("  - Representation: G = (V, E, I(C))")
    print("  - Status: [CONTEXT_CONDITIONED_EVALUATION]")
    print(f"  - Validation: Strictly evaluated via {n_splits}-fold cross-validation.")
    print("==================================================\n")


if __name__ == "__main__":
    run_milestone3_4_benchmark()

/tmp/ipykernel_974/1704609810.py:190: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(model_name, device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 3.4] Profiling & Extracting Vector Features for N=20 Tasks...


/tmp/ipykernel_974/1704609810.py:139: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  cos_sim = float(torch.dot(delta_a_vec, delta_b_vec) / (norm_a * norm_b))



🔄 Executing 5-Fold Cross Validation (Context-Conditioned Models)...

📊 MILESTONE 3.4: 5-FOLD CV BENCHMARK REPORT (N=20)
1. Additive Baseline (I_hat = 0):        MAE = 1.3481
2. Global Mean Baseline (I_hat = I_bar): MAE = 0.9596
3. Linear Context Model (I_hat = f(C)):  MAE = 0.4633
4. Shuffled-Label Control:               MAE = 1.0522
--------------------------------------------------
Improvement over Additive Baseline:      +65.6%
Improvement over Global Mean Baseline:   +51.7%
Sign Accuracy (Additive):                11/20 (55.0%)
Sign Accuracy (Context Model):           17/20 (85.0%)
EPISTEMIC METADATA:
  - Representation: G = (V, E, I(C))
  - Status: [CONTEXT_CONDITIONED_EVALUATION]
  - Validation: Strictly evaluated via 5-fold cross-validation.



In [17]:
"""
milestone3_5_feature_ablation.py
================================
Milestone 3.5 - Feature Ablation & Vector Geometry Benchmark (N=20, 5-Fold CV).

Evaluates 8 Model Variants (M0 - M7):
  M0: Additive Baseline (I_hat = 0)
  M1: Global Mean Baseline (I_hat = I_bar)
  M2: CleanDiff Only (I_hat = f(CleanDiff))
  M3: ResidNorm Only (I_hat = f(ResidNorm))
  M4: Vector Magnitudes Only (I_hat = f(||Delta_A||, ||Delta_B||))
  M5: Angular Geometry Only (I_hat = f(cos_theta))
  M6: Full Vector Geometry (I_hat = f(||Delta_A||, ||Delta_B||, cos_theta))
  M7: Full Context Model (All 5 Features)
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold


@dataclass
class ContextFeatureProfile:
    task_id: int
    prompt_text: str
    delta_a_scalar: float
    delta_b_scalar: float
    delta_ab_scalar: float
    i_ab_actual: float

    # Features
    clean_diff: float      # f0
    resid_norm: float      # f1
    norm_delta_a: float    # f2
    norm_delta_b: float    # f3
    cos_sim_delta: float   # f4

    def get_all_features(self) -> np.ndarray:
        return np.array([
            self.clean_diff,
            self.resid_norm,
            self.norm_delta_a,
            self.norm_delta_b,
            self.cos_sim_delta
        ], dtype=np.float64)


def generate_ioi_dataset_20(model: HookedTransformer) -> List[Dict]:
    """تولید ۲۰ مسئله IOI با سنجش دقیق تک‌توکنی اسامی در توکنایزر GPT-2."""
    candidate_pairs = [
        ("John", "Mary"), ("Alice", "Bob"), ("David", "Sarah"), ("Charlie", "Emma"),
        ("Paul", "Kate"), ("Mark", "Jane"), ("Luke", "Anne"), ("Tom", "Lucy"),
        ("Jack", "Amy"), ("Dan", "Eve"), ("Ben", "Grace"), ("Sam", "Mia"),
        ("Max", "Lia"), ("Leo", "Zoe"), ("Ray", "Joy"), ("Alex", "Kim"),
        ("Cole", "Fay"), ("Dean", "May"), ("Grant", "Hope"), ("Frank", "Ruth")
    ]

    valid_pairs = []
    for n1, n2 in candidate_pairs:
        tok_1 = model.to_tokens(f" {n1}", prepend_bos=False).squeeze()
        tok_2 = model.to_tokens(f" {n2}", prepend_bos=False).squeeze()
        if tok_1.ndim == 0 and tok_2.ndim == 0:
            valid_pairs.append((n1, n2))

    objects = ["drink", "toy", "book", "bag", "pen", "key", "apple", "letter", "box", "gift"]
    verbs = ["gave", "handed", "passed", "brought", "sent"]

    tasks = []
    for idx in range(min(20, len(valid_pairs))):
        n1, n2 = valid_pairs[idx]
        obj = objects[idx % len(objects)]
        verb = verbs[idx % len(verbs)]
        clean = f"Then, {n1} and {n2} went to the store. {n1} {verb} a {obj} to"
        corrupted = f"Then, {n1} and {n2} went to the store. {n2} {verb} a {obj} to"
        tasks.append({
            "clean": clean,
            "corrupted": corrupted,
            "correct": f" {n2}",
            "incorrect": f" {n1}"
        })
    return tasks


def compute_head_contrib(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


@torch.no_grad()
def extract_task_profile(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    task_id: int,
    pos: int = -1
) -> ContextFeatureProfile:
    l_tgt, _ = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    clean_logits, clean_cache = model.run_with_cache(clean_toks)
    base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).detach().cpu().item())
    clean_diff = float((clean_logits[0, pos, correct_id] - clean_logits[0, pos, incorrect_id]).detach().cpu().item())
    resid_norm = float(corrupted_cache[f"blocks.{l_tgt}.hook_resid_pre"][0, pos, :].norm().detach().cpu().item())

    # استخراج بردارهای Residual با حذف Gradient Tracking
    contrib_a_clean = compute_head_contrib(model, clean_cache, l_a, h_a, pos).detach()
    contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, l_a, h_a, pos).detach()
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contrib(model, clean_cache, l_b, h_b, pos).detach()
    contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, l_b, h_b, pos).detach()
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    norm_a = float(delta_a_vec.norm().cpu().item())
    norm_b = float(delta_b_vec.norm().cpu().item())

    if norm_a > 1e-6 and norm_b > 1e-6:
        cos_sim = float((torch.dot(delta_a_vec, delta_b_vec) / (norm_a * norm_b)).cpu().item())
    else:
        cos_sim = 0.0

    # مداخلات
    def hook_a(act, hook):
        act[0, pos, :] += delta_a_vec
        return act

    def hook_b(act, hook):
        act[0, pos, :] += delta_b_vec
        return act

    def hook_ab(act, hook):
        act[0, pos, :] += (delta_a_vec + delta_b_vec)
        return act

    out_a = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_a)])
    out_b = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_b)])
    out_ab = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_ab)])

    delta_a_s = float((out_a[0, pos, correct_id] - out_a[0, pos, incorrect_id]).detach().cpu().item()) - base_diff
    delta_b_s = float((out_b[0, pos, correct_id] - out_b[0, pos, incorrect_id]).detach().cpu().item()) - base_diff
    delta_ab_s = float((out_ab[0, pos, correct_id] - out_ab[0, pos, incorrect_id]).detach().cpu().item()) - base_diff

    i_ab = delta_ab_s - (delta_a_s + delta_b_s)

    return ContextFeatureProfile(
        task_id=task_id,
        prompt_text=task["clean"][:30] + "...",
        delta_a_scalar=delta_a_s,
        delta_b_scalar=delta_b_s,
        delta_ab_scalar=delta_ab_s,
        i_ab_actual=i_ab,
        clean_diff=clean_diff,
        resid_norm=resid_norm,
        norm_delta_a=norm_a,
        norm_delta_b=norm_b,
        cos_sim_delta=cos_sim
    )


def run_milestone3_5_ablation(
    model_name: str = "gpt2",
    source_a: Tuple[int, int] = (9, 9),
    source_b: Tuple[int, int] = (10, 7),
    target: Tuple[int, int] = (11, 10),
    n_splits: int = 5
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained(model_name, device=device)
    model.eval()

    tasks = generate_ioi_dataset_20(model)
    profiles: List[ContextFeatureProfile] = []

    print(f"\n⚡️ [MILESTONE 3.5] Extracting Features & Running Feature Ablation (N={len(tasks)} Tasks)...")

    for idx, task in enumerate(tasks):
        p = extract_task_profile(model, source_a, source_b, target, task, idx + 1)
        profiles.append(p)

    X_all = np.array([p.get_all_features() for p in profiles])
    y_actual = np.array([p.i_ab_actual for p in profiles])
    y_delta_ab = np.array([p.delta_ab_scalar for p in profiles])
    y_additive_pred = np.array([p.delta_a_scalar + p.delta_b_scalar for p in profiles])

    # نگاشت زیرمجموعه ویژگی‌ها برای هر مدل
    # Indices: 0: CleanDiff, 1: ResidNorm, 2: ||DeltaA||, 3: ||DeltaB||, 4: cos_sim
    feature_sets = {
        "M2: CleanDiff Only": [0],
        "M3: ResidNorm Only": [1],
        "M4: Vector Magnitudes (||dA||, ||dB||)": [2, 3],
        "M5: Angular Geometry (cos_theta)": [4],
        "M6: Full Vector Geometry (Magnitudes + cos)": [2, 3, 4],
        "M7: Full Context Model (All 5 Features)": [0, 1, 2, 3, 4]
    }

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    errors = {
        "M0: Additive Baseline (I_hat = 0)": [],
        "M1: Global Mean Baseline (I_hat = I_bar)": [],
        **{model_name: [] for model_name in feature_sets.keys()}
    }

    sign_acc = {model_name: 0 for model_name in errors.keys()}

    for train_idx, test_idx in kf.split(X_all):
        y_train, y_test = y_actual[train_idx], y_actual[test_idx]
        ab_test_actual = y_delta_ab[test_idx]
        add_pred_test = y_additive_pred[test_idx]

        # M0: Additive
        pred_ab_m0 = add_pred_test
        errors["M0: Additive Baseline (I_hat = 0)"].extend(np.abs(ab_test_actual - pred_ab_m0))
        sign_acc["M0: Additive Baseline (I_hat = 0)"] += np.sum(np.sign(pred_ab_m0) == np.sign(ab_test_actual))

        # M1: Global Mean
        mean_i = float(np.mean(y_train))
        pred_ab_m1 = add_pred_test + mean_i
        errors["M1: Global Mean Baseline (I_hat = I_bar)"].extend(np.abs(ab_test_actual - pred_ab_m1))
        sign_acc["M1: Global Mean Baseline (I_hat = I_bar)"] += np.sum(np.sign(pred_ab_m1) == np.sign(ab_test_actual))

        # M2 - M7: Feature Subsets
        for name, feats in feature_sets.items():
            X_tr_sub = X_all[train_idx][:, feats]
            X_te_sub = X_all[test_idx][:, feats]

            scaler = StandardScaler()
            X_tr_sc = scaler.fit_transform(X_tr_sub)
            X_te_sc = scaler.transform(X_te_sub)

            reg = LinearRegression()
            reg.fit(X_tr_sc, y_train)
            i_pred = reg.predict(X_te_sc)

            pred_ab = add_pred_test + i_pred
            errors[name].extend(np.abs(ab_test_actual - pred_ab))
            sign_acc[name] += np.sum(np.sign(pred_ab) == np.sign(ab_test_actual))

    mae_m0 = float(np.mean(errors["M0: Additive Baseline (I_hat = 0)"]))

    print("\n==================================================================================")
    print(f"📊 MILESTONE 3.5: FEATURE ABLATION REPORT (5-FOLD CV, N={len(tasks)})")
    print("==================================================================================")
    print(f"{'Model Variant':<45} | {'MAE':<7} | {'vs Additive (%)':<15} | {'Sign Acc':<10}")
    print("----------------------------------------------------------------------------------")

    for name, err_list in errors.items():
        mae = float(np.mean(err_list))
        imp = ((mae_m0 - mae) / mae_m0) * 100
        acc_str = f"{sign_acc[name]}/{len(tasks)} ({sign_acc[name]/len(tasks)*100:.0f}%)"
        print(f"{name:<45} | {mae:6.4f}  | {imp:+6.1f}%          | {acc_str}")

    print("==================================================================================")
    print("EPISTEMIC METADATA:")
    print("  - Representation: G = (V, E, I(C))")
    print("  - Status: [FEATURE_ABLATION_EVALUATED]")
    print("  - Scope: Out-of-sample evaluation strictly within 20 IOI prompts.")
    print("==================================================================================\n")


if __name__ == "__main__":
    run_milestone3_5_ablation()

/tmp/ipykernel_974/1498782042.py:185: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(model_name, device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 3.5] Extracting Features & Running Feature Ablation (N=20 Tasks)...

📊 MILESTONE 3.5: FEATURE ABLATION REPORT (5-FOLD CV, N=20)
Model Variant                                 | MAE     | vs Additive (%) | Sign Acc  
----------------------------------------------------------------------------------
M0: Additive Baseline (I_hat = 0)             | 1.3481  |   +0.0%          | 11/20 (55%)
M1: Global Mean Baseline (I_hat = I_bar)      | 0.9596  |  +28.8%          | 15/20 (75%)
M2: CleanDiff Only                            | 0.6562  |  +51.3%          | 18/20 (90%)
M3: ResidNorm Only                            | 0.9236  |  +31.5%          | 16/20 (80%)
M4: Vector Magnitudes (||dA||, ||dB||)        | 0.4723  |  +65.0%          | 16/20 (80%)
M5: Angular Geometry (cos_theta)              | 0.8411  |  +37.6%          | 14/20 (70%)
M6: Full Vector Geometry (Magnitudes + cos)   | 0.5088  |  +62.3%          | 17/20 (85%)
M7: Full Con

In [18]:
"""
milestone3_6_causal_scaling_surface.py
=====================================
Milestone 3.6 - Causal Magnitude Scaling Surface & Template Shift Benchmark.

Objectives:
1. Causal Scaling Test: Evaluates I_AB(alpha_A, alpha_B) across a 6x6 grid of
   scaling factors alpha_A, alpha_B in {0.0, 0.25, 0.50, 1.0, 1.5, 2.0}.
   Tests if I_AB is a smooth function of alpha_A * alpha_B (Causal Scaling Hypothesis)
   or stays constant / irregular (Proxy Hypothesis).
2. Out-of-Distribution Template Shift: Evaluates the trained context-conditioned
   model on prompts with novel syntactic structures (different sentence templates).
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score


# =============================================================================
# 1. Dataset Generation (Template Shift)
# =============================================================================

def generate_shifted_template_tasks(model: HookedTransformer) -> Dict[str, List[Dict]]:
    """تولید مسائل IOI با دو قالب نحوی کاملاً متفاوت جهت سنجش تعمیم‌پذیری."""
    candidate_pairs = [
        ("John", "Mary"), ("Alice", "Bob"), ("David", "Sarah"), ("Charlie", "Emma"),
        ("Paul", "Kate"), ("Mark", "Jane"), ("Luke", "Anne"), ("Tom", "Lucy"),
        ("Jack", "Amy"), ("Dan", "Eve")
    ]

    valid_pairs = []
    for n1, n2 in candidate_pairs:
        tok_1 = model.to_tokens(f" {n1}", prepend_bos=False).squeeze()
        tok_2 = model.to_tokens(f" {n2}", prepend_bos=False).squeeze()
        if tok_1.ndim == 0 and tok_2.ndim == 0:
            valid_pairs.append((n1, n2))

    objects = ["drink", "toy", "book", "bag", "pen", "key", "apple", "letter", "box", "gift"]

    # Template 1 (Original / In-Distribution)
    template_in = []
    for idx in range(len(valid_pairs)):
        n1, n2 = valid_pairs[idx]
        obj = objects[idx % len(objects)]
        clean = f"Then, {n1} and {n2} went to the store. {n1} gave a {obj} to"
        corrupted = f"Then, {n1} and {n2} went to the store. {n2} gave a {obj} to"
        template_in.append({"clean": clean, "corrupted": corrupted, "correct": f" {n2}", "incorrect": f" {n1}"})

    # Template 2 (Syntactic Shift 1: Clause Order & Verb Shift)
    template_shift_1 = []
    for idx in range(len(valid_pairs)):
        n1, n2 = valid_pairs[idx]
        obj = objects[idx % len(objects)]
        clean = f"Yesterday, while at the library, {n1} and {n2} met. {n1} handed the {obj} to"
        corrupted = f"Yesterday, while at the library, {n1} and {n2} met. {n2} handed the {obj} to"
        template_shift_1.append({"clean": clean, "corrupted": corrupted, "correct": f" {n2}", "incorrect": f" {n1}"})

    # Template 3 (Syntactic Shift 2: Passive / Descriptive Context)
    template_shift_2 = []
    for idx in range(len(valid_pairs)):
        n1, n2 = valid_pairs[idx]
        obj = objects[idx % len(objects)]
        clean = f"Inside the room, {n1} saw {n2} sitting nearby. {n1} passed a {obj} to"
        corrupted = f"Inside the room, {n1} saw {n2} sitting nearby. {n2} passed a {obj} to"
        template_shift_2.append({"clean": clean, "corrupted": corrupted, "correct": f" {n2}", "incorrect": f" {n1}"})

    return {
        "In-Distribution Template": template_in,
        "Shifted Template 1 (Library)": template_shift_1,
        "Shifted Template 2 (Room)": template_shift_2
    }


def compute_head_contrib(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


# =============================================================================
# 2. Causal Scaling Surface Extraction (6x6 Grid)
# =============================================================================

@torch.no_grad()
def extract_causal_scaling_surface(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    alpha_grid: List[float],
    pos: int = -1
) -> np.ndarray:
    """
    استخراج سطح برهم‌کنش I(alpha_A, alpha_B) روی شبکه 6x6 از ضرایب مقیاس.
    """
    l_tgt, _ = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    clean_logits, clean_cache = model.run_with_cache(clean_toks)
    base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).detach().cpu().item())

    contrib_a_clean = compute_head_contrib(model, clean_cache, l_a, h_a, pos).detach()
    contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, l_a, h_a, pos).detach()
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contrib(model, clean_cache, l_b, h_b, pos).detach()
    contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, l_b, h_b, pos).detach()
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    surface_i_ab = np.zeros((len(alpha_grid), len(alpha_grid)), dtype=np.float64)

    for i, a_a in enumerate(alpha_grid):
        for j, a_b in enumerate(alpha_grid):
            # 1. Delta_A scaled
            def hook_scaled_a(act, hook):
                act[0, pos, :] += a_a * delta_a_vec
                return act

            # 2. Delta_B scaled
            def hook_scaled_b(act, hook):
                act[0, pos, :] += a_b * delta_b_vec
                return act

            # 3. Joint scaled
            def hook_scaled_ab(act, hook):
                act[0, pos, :] += (a_a * delta_a_vec + a_b * delta_b_vec)
                return act

            out_a = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_scaled_a)])
            out_b = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_scaled_b)])
            out_ab = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_scaled_ab)])

            d_a_s = float((out_a[0, pos, correct_id] - out_a[0, pos, incorrect_id]).detach().cpu().item()) - base_diff
            d_b_s = float((out_b[0, pos, correct_id] - out_b[0, pos, incorrect_id]).detach().cpu().item()) - base_diff
            d_ab_s = float((out_ab[0, pos, correct_id] - out_ab[0, pos, incorrect_id]).detach().cpu().item()) - base_diff

            # Calculate I_AB for this scaling combination
            surface_i_ab[i, j] = d_ab_s - (d_a_s + d_b_s)

    return surface_i_ab


# =============================================================================
# 3. Main Benchmark Execution
# =============================================================================

def run_milestone3_6_benchmark(
    model_name: str = "gpt2",
    source_a: Tuple[int, int] = (9, 9),
    source_b: Tuple[int, int] = (10, 7),
    target: Tuple[int, int] = (11, 10)
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained(model_name, device=device)
    model.eval()

    alpha_grid = [0.0, 0.25, 0.50, 1.0, 1.5, 2.0]
    datasets = generate_shifted_template_tasks(model)

    print("\n==================================================")
    print("⚡️ [MILESTONE 3.6] CAUSAL SCALING SURFACE & TEMPLATE SHIFT BENCHMARK")
    print("==================================================")
    print(f"Target: L{target[0]}H{target[1]} | Sources: L{source_a[0]}H{source_a[1]} & L{source_b[0]}H{source_b[1]}")
    print(f"Scaling Grid: {alpha_grid} (6x6 = 36 points per task)")

    # -------------------------------------------------------------------------
    # PART 1: CAUSAL MAGNITUDE SCALING TEST
    # -------------------------------------------------------------------------
    print("\n1️⃣ CAUSAL SCALING SURFACE ANALYSIS (Evaluating on In-Distribution Tasks):")
    sample_task = datasets["In-Distribution Template"][0]
    scaling_surface = extract_causal_scaling_surface(model, source_a, source_b, target, sample_task, alpha_grid)

    print("\n  [Matrix representation of I_AB(alpha_A, alpha_B)]:")
    print("  alpha_B ->  " + "  ".join([f"{a:5.2f}" for a in alpha_grid]))
    for i, a_a in enumerate(alpha_grid):
        row_str = "  ".join([f"{scaling_surface[i, j]:+6.3f}" for j in range(len(alpha_grid))])
        print(f"  alpha_A={a_a:4.2f}: {row_str}")

    # Test Multiplicative Hypothesis: I_AB ~ c * (alpha_A * alpha_B)
    products = []
    i_vals = []
    for i, a_a in enumerate(alpha_grid):
        for j, a_b in enumerate(alpha_grid):
            products.append(a_a * a_b)
            i_vals.append(scaling_surface[i, j])

    products = np.array(products).reshape(-1, 1)
    i_vals = np.array(i_vals)

    reg_mult = LinearRegression()
    reg_mult.fit(products, i_vals)
    r2_mult = r2_score(i_vals, reg_mult.predict(products))

    print(f"\n  ★ Multiplicative Fit R^2 (I_AB ~ c * alpha_A * alpha_B): {r2_mult:6.4f}")
    if r2_mult > 0.80:
        print("  ★ Status: [CAUSAL_SCALING_CONFIRMED] - Interaction scales smoothly with intervention magnitude.")
    else:
        print("  ★ Status: [PROXY_BEHAVIOR_DETECTED] - Interaction does not scale smoothly; magnitudes are likely proxies.")

    # -------------------------------------------------------------------------
    # PART 2: OUT-OF-DISTRIBUTION TEMPLATE SHIFT EVALUATION
    # -------------------------------------------------------------------------
    print("\n--------------------------------------------------")
    print("2️⃣ OUT-OF-DISTRIBUTION TEMPLATE SHIFT EVALUATION")
    print("--------------------------------------------------")

    for template_name, tasks in datasets.items():
        errs_add = []
        errs_i_scaled = []

        for task in tasks:
            clean_toks = model.to_tokens(task["clean"])
            corrupted_toks = model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = model.run_with_cache(clean_toks)
                base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

            correct_id = int(model.to_single_token(task["correct"]))
            incorrect_id = int(model.to_single_token(task["incorrect"]))
            base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).detach().cpu().item())

            contrib_a_clean = compute_head_contrib(model, clean_cache, source_a[0], source_a[1]).detach()
            contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, source_a[0], source_a[1]).detach()
            delta_a_vec = contrib_a_clean - contrib_a_corrupt

            contrib_b_clean = compute_head_contrib(model, clean_cache, source_b[0], source_b[1]).detach()
            contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, source_b[0], source_b[1]).detach()
            delta_b_vec = contrib_b_clean - contrib_b_corrupt

            def hook_a(act, hook):
                act[0, -1, :] += delta_a_vec
                return act

            def hook_b(act, hook):
                act[0, -1, :] += delta_b_vec
                return act

            def hook_ab(act, hook):
                act[0, -1, :] += (delta_a_vec + delta_b_vec)
                return act

            with torch.no_grad():
                out_a = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.hook_resid_pre", hook_a)])
                out_b = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.hook_resid_pre", hook_b)])
                out_ab = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.hook_resid_pre", hook_ab)])

            d_a = float((out_a[0, -1, correct_id] - out_a[0, -1, incorrect_id]).detach().cpu().item()) - base_diff
            d_b = float((out_b[0, -1, correct_id] - out_b[0, -1, incorrect_id]).detach().cpu().item()) - base_diff
            actual_ab = float((out_ab[0, -1, correct_id] - out_ab[0, -1, incorrect_id]).detach().cpu().item()) - base_diff

            pred_add = d_a + d_b
            # Predict I_AB using the causal scaling relation: I_AB = c * ||dA|| * ||dB||
            norm_a = float(delta_a_vec.norm().cpu().item())
            norm_b = float(delta_b_vec.norm().cpu().item())
            pred_i = 0.005 * norm_a * norm_b  # Scaled physics approximation
            pred_joint = pred_add + pred_i

            errs_add.append(abs(actual_ab - pred_add))
            errs_i_scaled.append(abs(actual_ab - pred_joint))

        mae_add = float(np.mean(errs_add))
        mae_scaled = float(np.mean(errs_i_scaled))
        imp_pct = ((mae_add - mae_scaled) / mae_add) * 100

        print(f"  [{template_name}] (N={len(tasks)}):")
        print(f"     ├── Additive MAE:         {mae_add:6.4f}")
        print(f"     ├── Causal Scaled MAE:    {mae_scaled:6.4f}")
        print(f"     └── Error Improvement:   {imp_pct:+5.1f}%")

    print("\n==================================================")
    print("EPISTEMIC METADATA:")
    print("  - Status: [CAUSAL_SCALING_SURFACE_EVALUATED]")
    print(f"  - Multiplicative Fit R^2: {r2_mult:6.4f}")
    print("==================================================\n")


if __name__ == "__main__":
    run_milestone3_6_benchmark()

/tmp/ipykernel_974/3275671615.py:169: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(model_name, device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 3.6] CAUSAL SCALING SURFACE & TEMPLATE SHIFT BENCHMARK
Target: L11H10 | Sources: L9H9 & L10H7
Scaling Grid: [0.0, 0.25, 0.5, 1.0, 1.5, 2.0] (6x6 = 36 points per task)

1️⃣ CAUSAL SCALING SURFACE ANALYSIS (Evaluating on In-Distribution Tasks):

  [Matrix representation of I_AB(alpha_A, alpha_B)]:
  alpha_B ->   0.00   0.25   0.50   1.00   1.50   2.00
  alpha_A=0.00: +0.000  +0.000  +0.000  +0.000  +0.000  +0.000
  alpha_A=0.25: +0.000  -0.023  -0.020  +0.051  +0.144  +0.207
  alpha_A=0.50: +0.000  -0.064  -0.084  +0.019  +0.204  +0.346
  alpha_A=1.00: +0.000  -0.154  -0.261  -0.239  +0.048  +0.372
  alpha_A=1.50: +0.000  -0.215  -0.401  -0.554  -0.344  +0.081
  alpha_A=2.00: +0.000  -0.242  -0.470  -0.762  -0.724  -0.375

  ★ Multiplicative Fit R^2 (I_AB ~ c * alpha_A * alpha_B): 0.1719
  ★ Status: [PROXY_BEHAVIOR_DETECTED] - Interaction does not scale smoothly; magnitudes are likely proxies.

---------------------------

In [19]:
"""
milestone3_7_internal_head_decomposition.py
===========================================
Milestone 3.7 - Internal Head Attribution & LayerNorm/Attention Disambiguation.

Decomposes the non-linear interaction I_AB(alpha_A, alpha_B) at Target L11H10 into:
  1. LayerNorm Scaling Factor Change: Measures denominator expansion in LN1 of Layer 11.
  2. Attention Pattern Mediation (A-clamp): Effect of patching Resid_pre while clamping
     Attention Pattern to corrupted vs clean.
  3. Value Vector Mediation (V-clamp): Effect of patching Resid_pre while clamping
     Value vectors to corrupted vs clean.
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer


@dataclass
class HeadDecompositionResult:
    alpha_a: float
    alpha_b: float
    total_interaction: float

    # LayerNorm
    ln_scale_baseline: float
    ln_scale_patched: float
    ln_scale_ratio: float

    # Internal Head Components
    effect_pattern_only: float    # Δ Logit Diff when ONLY Attention Pattern changes
    effect_value_only: float      # Δ Logit Diff when ONLY Value Vector changes
    effect_clamped_pattern: float # Δ Logit Diff when Resid_pre is patched but Pattern is locked
    effect_clamped_value: float   # Δ Logit Diff when Resid_pre is patched but Value is locked


def compute_head_contrib(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


@torch.no_grad()
def decompose_l11h10_interaction(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    alpha_a: float = 1.0,
    alpha_b: float = 1.0,
    pos: int = -1
) -> HeadDecompositionResult:
    l_tgt, h_tgt = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    clean_logits, clean_cache = model.run_with_cache(clean_toks)
    base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).detach().cpu().item())

    # -------------------------------------------------------------------------
    # 1. Residual Deltas from Sources
    # -------------------------------------------------------------------------
    contrib_a_clean = compute_head_contrib(model, clean_cache, l_a, h_a, pos).detach()
    contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, l_a, h_a, pos).detach()
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contrib(model, clean_cache, l_b, h_b, pos).detach()
    contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, l_b, h_b, pos).detach()
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    patched_vec = alpha_a * delta_a_vec + alpha_b * delta_b_vec

    def patch_resid_hook(act, hook):
        act[0, pos, :] += patched_vec
        return act

    # -------------------------------------------------------------------------
    # 2. Measure LayerNorm Scaling Factor
    # -------------------------------------------------------------------------
    # LayerNorm scale in TransformerLens is stored in hook_scale for LN1
    ln_scale_corrupted = float(corrupted_cache[f"blocks.{l_tgt}.ln1.hook_scale"][0, pos, 0].detach().cpu().item())

    scale_container = [None]
    def capture_ln_scale(act, hook):
        scale_container[0] = float(act[0, pos, 0].detach().cpu().item())
        return act

    model.run_with_hooks(
        corrupted_toks,
        fwd_hooks=[
            (f"blocks.{l_tgt}.hook_resid_pre", patch_resid_hook),
            (f"blocks.{l_tgt}.ln1.hook_scale", capture_ln_scale)
        ]
    )
    ln_scale_patched = scale_container[0]
    ln_ratio = ln_scale_patched / (ln_scale_corrupted + 1e-8)

    # -------------------------------------------------------------------------
    # 3. Disambiguate Attention Pattern (A) vs Value Vector (V)
    # -------------------------------------------------------------------------
    corrupted_pattern = corrupted_cache[f"blocks.{l_tgt}.attn.hook_pattern"][0, h_tgt, :, :].clone().detach()
    corrupted_v = corrupted_cache[f"blocks.{l_tgt}.attn.hook_v"][0, pos, h_tgt, :].clone().detach()

    # Hook A-clamp: Lock Attention Pattern to corrupted baseline during patched forward
    def clamp_pattern_hook(act, hook):
        act[0, h_tgt, :, :] = corrupted_pattern
        return act

    # Hook V-clamp: Lock Value Vector of H_target to corrupted baseline during patched forward
    def clamp_v_hook(act, hook):
        act[0, pos, h_tgt, :] = corrupted_v
        return act

    # A) Full Joint Run
    out_joint = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", patch_resid_hook)])
    diff_joint = float((out_joint[0, pos, correct_id] - out_joint[0, pos, incorrect_id]).detach().cpu().item()) - base_diff

    # B) Run with Resid Patched + Pattern Clamped (Isolates V / Resid effect)
    out_clamp_pattern = model.run_with_hooks(
        corrupted_toks,
        fwd_hooks=[
            (f"blocks.{l_tgt}.hook_resid_pre", patch_resid_hook),
            (f"blocks.{l_tgt}.attn.hook_pattern", clamp_pattern_hook)
        ]
    )
    diff_clamp_pattern = float((out_clamp_pattern[0, pos, correct_id] - out_clamp_pattern[0, pos, incorrect_id]).detach().cpu().item()) - base_diff

    # C) Run with Resid Patched + Value Clamped (Isolates Pattern effect)
    out_clamp_v = model.run_with_hooks(
        corrupted_toks,
        fwd_hooks=[
            (f"blocks.{l_tgt}.hook_resid_pre", patch_resid_hook),
            (f"blocks.{l_tgt}.attn.hook_v", clamp_v_hook)
        ]
    )
    diff_clamp_v = float((out_clamp_v[0, pos, correct_id] - out_clamp_v[0, pos, incorrect_id]).detach().cpu().item()) - base_diff

    # Single interventions for calculating I_AB
    def hook_single_a(act, hook):
        act[0, pos, :] += alpha_a * delta_a_vec
        return act

    def hook_single_b(act, hook):
        act[0, pos, :] += alpha_b * delta_b_vec
        return act

    out_a = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_single_a)])
    out_b = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_single_b)])

    diff_a = float((out_a[0, pos, correct_id] - out_a[0, pos, incorrect_id]).detach().cpu().item()) - base_diff
    diff_b = float((out_b[0, pos, correct_id] - out_b[0, pos, incorrect_id]).detach().cpu().item()) - base_diff

    total_i_ab = diff_joint - (diff_a + diff_b)

    return HeadDecompositionResult(
        alpha_a=alpha_a,
        alpha_b=alpha_b,
        total_interaction=total_i_ab,
        ln_scale_baseline=ln_scale_corrupted,
        ln_scale_patched=ln_scale_patched,
        ln_scale_ratio=ln_ratio,
        effect_pattern_only=diff_joint - diff_clamp_pattern,
        effect_value_only=diff_joint - diff_clamp_v,
        effect_clamped_pattern=diff_clamp_pattern,
        effect_clamped_value=diff_clamp_v
    )


def run_milestone3_7_decomposition_suite(
    model_name: str = "gpt2",
    source_a: Tuple[int, int] = (9, 9),
    source_b: Tuple[int, int] = (10, 7),
    target: Tuple[int, int] = (11, 10)
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained(model_name, device=device)
    model.eval()

    sample_task = {
        "clean": "Then, John and Mary went to the store. John gave a drink to",
        "corrupted": "Then, John and Mary went to the store. Mary gave a drink to",
        "correct": " Mary",
        "incorrect": " John"
    }

    scaling_pairs = [(1.0, 1.0), (1.5, 1.0), (1.0, 1.5), (2.0, 1.0), (1.0, 2.0), (2.0, 2.0)]

    print("\n==================================================")
    print("🔬 MILESTONE 3.7: L11H10 INTERNAL DECOMPOSITION REPORT")
    print("==================================================")
    print(f"Target Head: L{target[0]}H{target[1]} | Sources: L{source_a[0]}H{source_a[1]} & L{source_b[0]}H{source_b[1]}\n")

    print(f"{'Scale (aA, aB)':<15} | {'Total I_AB':<10} | {'LN Scale Ratio':<15} | {'Pattern Contrib':<15} | {'Value Contrib':<15}")
    print("-" * 80)

    for a_a, a_b in scaling_pairs:
        res = decompose_l11h10_interaction(model, source_a, source_b, target, sample_task, alpha_a=a_a, alpha_b=a_b)
        scale_str = f"({a_a:.1f}, {a_b:.1f})"
        print(f"{scale_str:<15} | {res.total_interaction:+9.4f} | {res.ln_scale_ratio:14.4f} | {res.effect_pattern_only:+14.4f} | {res.effect_value_only:+14.4f}")

    print("==================================================")
    print("EPISTEMIC INTERPRETATION GUIDELINE:")
    print("  - LN Scale Ratio < 1.0: LayerNorm denominator expanded, suppressing net head output.")
    print("  - Pattern Contrib dominant: Non-linearity originates in Q/K Attention Softmax shift.")
    print("  - Value Contrib dominant: Non-linearity originates in Residual Stream V-vector mixing.")
    print("==================================================\n")


if __name__ == "__main__":
    run_milestone3_7_decomposition_suite()

/tmp/ipykernel_974/24117353.py:186: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(model_name, device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

🔬 MILESTONE 3.7: L11H10 INTERNAL DECOMPOSITION REPORT
Target Head: L11H10 | Sources: L9H9 & L10H7

Scale (aA, aB)  | Total I_AB | LN Scale Ratio  | Pattern Contrib | Value Contrib  
--------------------------------------------------------------------------------
(1.0, 1.0)      |   -0.2391 |         1.0062 |        +0.0249 |        +0.0000
(1.5, 1.0)      |   -0.5539 |         1.0069 |        -0.5119 |        +0.0032
(1.0, 1.5)      |   +0.0483 |         1.0137 |        +0.6547 |        -0.0023
(2.0, 1.0)      |   -0.7621 |         1.0107 |        -0.8601 |        +0.0056
(1.0, 2.0)      |   +0.3725 |         1.0239 |        +1.1735 |        -0.0030
(2.0, 2.0)      |   -0.3747 |         1.0210 |        +0.0399 |        +0.0001
EPISTEMIC INTERPRETATION GUIDELINE:
  - LN Scale Ratio < 1.0: LayerNorm denominator expanded, suppressing net head output.
  - Pattern Contrib dominant: Non-linearity originates in Q/K Attention Softmax shift.


In [ ]:
"""
milestone3_7_attention_entropy.py
=================================
Calculates the Shannon Entropy shift of target head L11H10's attention pattern
under interventions alpha_A and alpha_B from source heads L9H9 and L10H7.
"""

import torch
import numpy as np
from typing import Dict, Tuple
from transformer_lens import HookedTransformer


def compute_head_contrib(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def calculate_pattern_entropy(pattern_vec: torch.Tensor, eps: float = 1e-12) -> float:
    """محاسبه آنتروپی شانون (پایه ۲) برای یک توزیع احتمال توجه."""
    probs = pattern_vec.detach().cpu()
    probs = torch.clamp(probs, min=eps)
    entropy = -torch.sum(probs * torch.log2(probs))
    return float(entropy.item())


@torch.no_grad()
def measure_entropy_shift(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    alpha_grid: list,
    pos: int = -1
):
    l_tgt, h_tgt = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    _, clean_cache = model.run_with_cache(clean_toks)
    base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).detach().cpu().item())

    # استخراج بردارهای مداخله
    contrib_a_clean = compute_head_contrib(model, clean_cache, l_a, h_a, pos).detach()
    contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, l_a, h_a, pos).detach()
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contrib(model, clean_cache, l_b, h_b, pos).detach()
    contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, l_b, h_b, pos).detach()
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    # آنتروپی پایه در حالت Corrupted بدون مداخله
    base_pattern = corrupted_cache[f"blocks.{l_tgt}.attn.hook_pattern"][0, h_tgt, pos, :]
    base_entropy = calculate_pattern_entropy(base_pattern)

    print(f"📌 Base Attention Entropy (Corrupted, No Intervention): {base_entropy:6.4f} bits\n")
    print(f"{'Scale (aA, aB)':<15} | {'Delta Logit Diff':<18} | {'Attn Entropy (bits)':<20} | {'Entropy Shift (ΔH)':<20}")
    print("-" * 80)

    for a_a in alpha_grid:
        for a_b in alpha_grid:
            patched_vec = a_a * delta_a_vec + a_b * delta_b_vec

            def patch_resid_hook(act, hook):
                act[0, pos, :] += patched_vec
                return act

            pattern_container = [None]
            def capture_pattern(act, hook):
                pattern_container[0] = act[0, h_tgt, pos, :].clone().detach()
                return act

            out = model.run_with_hooks(
                corrupted_toks,
                fwd_hooks=[
                    (f"blocks.{l_tgt}.hook_resid_pre", patch_resid_hook),
                    (f"blocks.{l_tgt}.attn.hook_pattern", capture_pattern)
                ]
            )

            logit_diff = float((out[0, pos, correct_id] - out[0, pos, incorrect_id]).detach().cpu().item()) - base_diff
            curr_pattern = pattern_container[0]
            curr_entropy = calculate_pattern_entropy(curr_pattern)
            delta_entropy = curr_entropy - base_entropy

            scale_str = f"({a_a:.2f}, {a_b:.2f})"
            print(f"{scale_str:<15} | {logit_diff:+17.4f} | {curr_entropy:19.4f} | {delta_entropy:+19.4f}")


def run_entropy_benchmark(
    model_name: str = "gpt2",
    source_a: Tuple[int, int] = (9, 9),
    source_b: Tuple[int, int] = (10, 7),
    target: Tuple[int, int] = (11, 10)
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained(model_name, device=device)
    model.eval()

    sample_task = {
        "clean": "Then, John and Mary went to the store. John gave a drink to",
        "corrupted": "Then, John and Mary went to the store. Mary gave a drink to",
        "correct": " Mary",
        "incorrect": " John"
    }

    alpha_grid = [0.0, 0.5, 1.0, 1.5, 2.0]

    print("\n==================================================")
    print("🧪 ATTENTION PATTERN ENTROPY BENCHMARK (L11H10)")
    print("==================================================")

    measure_entropy_shift(model, source_a, source_b, target, sample_task, alpha_grid)

    print("\n==================================================")
    print("EPISTEMIC INTERPRETATION:")
    print("  - ΔH < 0: Softmax saturation / Head focuses sharply on fewer tokens.")
    print("  - ΔH > 0: Diffused attention / Head distributes attention across multiple tokens.")
    print("==================================================\n")


if __name__ == "__main__":
    run_entropy_benchmark()